# 네이버 뉴스·댓글 전수 수집기

**사용법**
1. 처음 한 번만: 아래 **① 설치** 칸을 실행하고, 커널을 다시 시작합니다.
2. **② 수집 설정** 칸에서 검색어·언론사·기간·저장 폴더를 고칩니다.
3. 상단의 **Run All(모두 실행)** 을 누릅니다. 끝나면 저장 폴더에 `수집보고서.txt` 가 생깁니다.

**알아둘 것**
- 중간에 멈추거나 네이버가 잠시 막아도 괜찮습니다. **같은 설정으로 다시 Run All** 하면 끝난 부분은 건너뛰고 이어서 받습니다.
- 댓글은 **원댓글 + 답글 전부**(삭제·클린봇·운영규정 미준수·권리침해 포함)를 받고, 기사마다 네이버가 집계한 댓글 수와 비교해 보고서에 적습니다.
- `수집방식 = "전량"` (기본): 기간 내 모든 기사를 번호 순서대로 열어 본문·사진설명까지 확인합니다. **빠짐이 없고**, 네이버 검색을 쓰지 않아 **403 차단이 거의 없습니다**. 받은 전체 기사는 `_cache` 폴더에 남아, **다음에 검색어만 바꿔 돌리면 다시 받지 않고 몇 분 만에** 끝납니다.
  - 걸리는 시간(참고): 경향신문 7년 ≈ 56만 건 ≈ 3~4시간, 연합뉴스 7년 ≈ 580만 건 ≈ 1.5~2일. 디스크는 1만 건당 약 12MB(경향 7년 ≈ 0.7GB).
- `수집방식 = "검색"`: 네이버 검색 결과 기반. 짧은 기간을 빨리 확인할 때만 권장합니다. 검색 서버는 요청이 많으면 403으로 막고, **사진 설명에만 검색어가 있는 기사**는 빠집니다.

**저장되는 파일** (저장폴더/검색어/…)
- `언론사/articles.csv` (+ 엑셀용 `.xlsx`): 기사 1건 = 1행
- `comments/검색어_언론사.csv` (+ `.xlsx`): 댓글·답글 1건 = 1행, 기사와는 `url` 로 연결
- `comments/check_검색어_언론사.csv`: 기사별 네이버 집계 대비 수집 수
- `수집보고서.txt / .csv`: 전체 요약


In [5]:
# ① 설치 — 처음 한 번만 실행하고, 끝나면 커널을 다시 시작하세요
%pip install -q requests beautifulsoup4 pandas tqdm xlsxwriter ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# ② 수집 설정 ★ 이 칸만 고치세요 ★
검색어   = ["5.18", "5·18"]                  # 여러 개: ["5.18", "5·18"] / 빈 목록 [] 이면 검색어 없이 언론사 전체 기사
언론사   = ["경향신문", "국민일보", "동아일보", "서울신문", "조선일보", "중앙일보", "한겨레", "한국일보", "매일경제", "서울경제", "아시아경제", "조선비즈", "한국경제", "헤럴드경제", "파이낸셜뉴스", "KBS", "MBC", "SBS", 
         "JTBC", "채널A", "MBN", "TV조선", "연합뉴스", "뉴시스", "뉴스1", "연합뉴스TV", "YTN", "이데일리", "머니투데이", "서울신문", "문화일보", "세계일보"]    # 이름 목록, 또는 "전체" (79곳). 가능한 이름은 아래 코드 칸의 NEWS_OFFICES 참고
시작일   = "2019-01-01"
종료일   = "2019-01-31"
저장폴더 = r"C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722"

수집방식 = "검색"    # "전량": 빠짐 없음·차단 거의 없음(긴 기간·여러 언론사는 이것) / "검색": 짧은 기간 빠른 확인용
댓글수집 = True      # True: 댓글 + 답글 전수 / False: 기사만

# ── 고급 설정 (보통은 그대로) ──
요청간격_초 = 2.0    # "검색" 방식에서만 쓰는 검색 요청 간격. 403 차단이 자주 나면 3~4로
엄격필터   = True    # True: 제목·부제·본문·사진설명에 검색어가 글자 그대로 있는 기사만 저장
이미지수집 = True # True: 기사 이미지 주소 저장

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 네이버 뉴스·댓글 전수 수집기 — 코드 (수정하지 마세요. 설정은 위 '수집 설정' 칸에서)
# ══════════════════════════════════════════════════════════════════════════════
import os, time, re, json
import requests
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

# ══════════════════════════════════════════════════════════════════════════════
# 출력 표시 개선: 키워드별 색 태그 + 키워드별로 분리된 진행바
#   · 모든 print 줄 앞에 [키워드] 태그가 색깔로 붙는다 (병렬 실행 시 구분용)
#   · 진행바는 tqdm.auto 를 써서 Jupyter/VS Code 에서는 위젯 막대로 표시된다.
#     막대마다 독립된 칸이라 두 키워드의 막대가 한 줄에서 뒤엉키지 않는다.
#     (위젯이 안 보이면: pip install ipywidgets 후 커널 재시작)
#   · 이 블록은 반드시 import 셀(from tqdm import tqdm) "뒤"에 실행돼야 한다.
# ══════════════════════════════════════════════════════════════════════════════
import threading as _threading, builtins as _builtins
from tqdm.auto import tqdm as _tqdm_auto

_CTX = _threading.local()
_PRINT_LOCK = _threading.Lock()
_KW_COLORS = ["\033[1;34m", "\033[1;35m", "\033[1;32m", "\033[1;33m", "\033[1;36m", "\033[1;31m"]
_KW_COLOR_MAP = {}

def _kw_tag(color=True):
    kw = getattr(_CTX, "kw", None)
    if not kw:
        return ""
    if not color:
        return f"[{kw}] "
    c = _KW_COLOR_MAP.setdefault(kw, _KW_COLORS[len(_KW_COLOR_MAP) % len(_KW_COLORS)])
    return f"{c}[{kw}]\033[0m "

def print(*args, sep=" ", end="\n", **kwargs):
    """현재 스레드의 키워드를 각 줄 앞에 붙여 출력하는 print."""
    tag = _kw_tag()
    if tag and kwargs.get("file") is None:
        text = sep.join(str(a) for a in args)
        text = "\n".join((tag + ln) if ln.strip() else ln for ln in text.split("\n"))
        args, sep = (text,), " "
    with _PRINT_LOCK:
        _builtins.print(*args, sep=sep, end=end, **kwargs)

def tqdm(*args, **kwargs):
    """desc 앞에 [키워드]를 붙이고, 위젯 막대(tqdm.auto)로 표시하는 tqdm."""
    kwargs["desc"] = _kw_tag(color=False) + kwargs.get("desc", "")
    for k in ("bar_format", "ncols", "position"):
        kwargs.pop(k, None)
    kwargs.setdefault("leave", True)
    return _tqdm_auto(*args, **kwargs)

def _with_kw(kw, fn, *args):
    """스레드풀 작업자에게도 키워드 태그를 넘겨주는 래퍼."""
    _CTX.kw = kw
    try:
        return fn(*args)
    finally:
        _CTX.kw = None


# 디렉토리 생성 함수
def create_output_dir(base_dir, sub_dir):
    output_dir = os.path.join(base_dir, sub_dir)
    os.makedirs(output_dir, exist_ok=True)
    return output_dir


# ── 이어하기(resume) 지원 ────────────────────────────────────────────────────
# 중간에 중단(오류/전원차단/커널중단 등)되더라도 재실행 시 이미 완료된 단위
# (언론사 1곳의 기사수집, 언론사 1곳의 댓글수집)는 건너뛰고 이어서 진행하도록
# 한다. 핵심은 "결과 파일이 최종 경로에 존재 = 그 단위는 완전히 끝난 것"이
# 항상 참이어야 한다는 것이라, to_csv 를 임시파일에 쓴 뒤 os.replace 로
# 원자적으로 옮긴다(도중에 중단돼도 최종 경로에는 반쪽짜리 파일이 남지 않음).
def _atomic_write_csv(df, path, **kwargs):
    tmp_path = path + ".tmp"
    df.to_csv(tmp_path, **kwargs)
    os.replace(tmp_path, path)


def _atomic_write_text(path, text):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp_path, path)

# 이미지 화질 조정 함수
def adjust_image_quality(url, quality='medium'):
    """
    네이버 이미지 URL의 화질을 조정하는 함수

    Args:
        url: 원본 이미지 URL
        quality: 'high' (원본), 'medium' (w860), 'low' (w647)

    Returns:
        str: 화질이 조정된 이미지 URL
    """
    if not url or 'imgnews.pstatic.net' not in url:
        return url

    try:
        parsed = urlparse(url)
        query_params = parse_qs(parsed.query)

        # 화질 설정
        if quality == 'high':
            # 원본 화질 - type 파라미터 제거
            query_params.pop('type', None)
        elif quality == 'medium':
            query_params['type'] = ['w860']
        elif quality == 'low':
            query_params['type'] = ['w647']

        # URL 재구성
        new_query = urlencode(query_params, doseq=True)
        new_parsed = parsed._replace(query=new_query)
        return urlunparse(new_parsed)

    except Exception:
        return url

# 네이버 뉴스 링크 수집 함수 (날짜 제한 개선)
# ── 검색어 기반 기사 링크 수집 (2026.09 네이버 검색 구조 변경 대응판 v3) ────────
# 구조:
#   · 네이버 뉴스 검색은 무한 스크롤. 첫 화면 HTML 안의 more API 주소(…/api/tab/more?…)를
#     따라가며 "url"이 빌 때까지 넘긴다.
#   · 한 번의 검색은 약 600건(60여 페이지)에서 끊긴다(실측) → 구간이 그 근처까지 차면
#     구간을 반으로 나눠 다시 검색한다(적응형 구간: 30일 → 15일 → … → 1일).
#     기사가 적은 기간은 30일 1회로 끝나므로 1일 단위 고정보다 요청이 훨씬 적다.
#   · '관련뉴스 N건 전체보기' 묶음 안에 접힌 기사도 연다.
#   · 차단(403/429)되면 모든 스레드가 함께 쉬었다가(1→3→5→10→15→30분) 재시도한다.
#     그래도 안 풀린 구간은 건너뛰지 않고 '미완료'로 남겨, 다시 실행하면 이어서 받는다.
#   · 끝난 구간은 {언론사 폴더}/_links_progress.json 에 저장 → 중단 후 이어하기.
import re, time, json, random, threading, requests
from datetime import datetime, timedelta

SEARCH_DELAY = 2.0            # 요청 사이 기본 간격(초). 실제로는 0.7~1.4배로 흔들어 보낸다
SEARCH_WINDOW_DAYS = 1        # 검색 구간 길이(일). 1 = 하루씩(권장, 누락 최소). 2 이상이면 결과가 많을 때 자동으로 반씩 쪼갬
SEARCH_SPLIT_PAGES = 40       # 한 구간이 이 페이지 수(≈400건) 이상이면 반으로 나눈다 (상한 ≈62쪽)
BLOCK_WAITS = [60, 180, 300, 600, 900, 1800]   # 차단 시 단계별 대기(초), 합계 약 64분
SEARCH_DIRECT = True          # True: 무거운 검색 첫 화면(약 350KB)을 건너뛰고 more API를 처음부터 바로 호출
                              #       (2026.09.30 실측: 결과 동일 범위, 요청 1회당 크기·부담 감소). 실패하면 자동으로 첫 화면 방식으로 돌아감
_MORE_BASE = "https://s.search.naver.com/p/newssearch/3/api/tab/more"
# 브라우저(Chrome 154)가 실제로 보내는 요청 머리글 — 없으면 봇으로 분류될 수 있어 함께 보낸다
_BROWSER_HDRS = {
    "sec-ch-ua": '"Chromium";v="154", "Google Chrome";v="154", "Not?A_Brand";v="99"',
    "sec-ch-ua-mobile": "?0", "sec-ch-ua-platform": '"Windows"',
}


def _fetch_hdrs(url):
    """요청 종류별 브라우저 머리글: 검색 첫 화면(문서 이동) / more API(같은 사이트 XHR)."""
    if "/api/tab/more" in url:
        return {**_BROWSER_HDRS, "Accept": "application/json, text/plain, */*",
                "Sec-Fetch-Dest": "empty", "Sec-Fetch-Mode": "cors", "Sec-Fetch-Site": "same-site"}
    return {**_BROWSER_HDRS,
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
            "Sec-Fetch-Dest": "document", "Sec-Fetch-Mode": "navigate", "Sec-Fetch-Site": "same-origin",
            "Sec-Fetch-User": "?1", "Upgrade-Insecure-Requests": "1"}

_NAVER_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
             "(KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36")
_NEWS_LINK_RE = re.compile(r"https://n\.news\.naver\.com/mnews/article/\d{3}/\d{10}(?:\?sid=\d+)?")
_MORE_API_RE = re.compile(
    r"https:\\?/\\?/s\.search\.naver\.com\\?/p\\?/newssearch\\?/\d+\\?/api\\?/tab\\?/more\?[^\"'\s<>]+")
_RELATED_RE = re.compile(r"\?[^\"'\s<>\\]*?related=1[^\"'\s<>\\]*")

_NAVER_LOCK = threading.Lock()
_NAVER_COOLDOWN = [0.0]       # 모든 스레드가 공유: 이 시각(time.time())까지는 요청하지 않는다


class LinkCollectionIncomplete(Exception):
    """차단이 끝내 풀리지 않아 일부 구간을 받지 못했을 때."""
    def __init__(self, links, failed):
        super().__init__(f"{len(failed)}개 구간 미완료")
        self.links, self.failed = links, failed


def _unescape_url(u):
    return (u.replace("\\/", "/").replace("\\u0026", "&")
             .replace("\\u003d", "=").replace("&amp;", "&"))


# ★ 스포츠·연예 기사는 검색 결과에서 m.sports.naver.com / m.entertain.naver.com 주소로 걸려 있다
#   (2026.09 실측). 이전에는 n.news 주소만 인식해 스포츠·연예 기사가 한 건도 수집되지 않았다.
#   → 이런 주소도 인식해 표준 기사 주소(n.news.naver.com/mnews/article/언론사/기사번호)로 바꾼다.
_SE_LINK_RE = re.compile(
    r"https?://(?:m\.)?(?:sports|entertain)\.naver\.com/(?:[a-z]+/)?article/(\d{3})/(\d{10})")


def _extract_links(text):
    text = text.replace("\\/", "/").replace("\\u0026", "&").replace("&amp;", "&")
    links = _NEWS_LINK_RE.findall(text)
    links += [f"https://n.news.naver.com/mnews/article/{o}/{a}" for o, a in _SE_LINK_RE.findall(text)]
    return list(dict.fromkeys(links))


def _extract_related(text):
    """'관련뉴스 N건 전체보기' 묶음 주소(…docid=…&related=1…)를 docid 기준으로 모은다."""
    text = text.replace("\\/", "/").replace("\\u0026", "&").replace("&amp;", "&")
    out = {}
    for q in _RELATED_RE.findall(text):
        m = re.search(r"docid=(\d+)", q)
        if m:
            out.setdefault(m.group(1), q)
    return out


def _polite_sleep(delay=None):
    time.sleep((SEARCH_DELAY if delay is None else delay) * random.uniform(0.7, 1.4))


def _wait_cooldown():
    with _NAVER_LOCK:
        rem = _NAVER_COOLDOWN[0] - time.time()
    if rem > 0:
        time.sleep(rem)


def _get_with_retry(session, url, params=None):
    """정상 응답이 올 때까지 재시도. 차단되면 모든 스레드가 함께 단계적으로 쉰다."""
    status = None
    for attempt in range(len(BLOCK_WAITS) + 1):
        _wait_cooldown()
        try:
            r = session.get(url, params=params, headers=_fetch_hdrs(url), timeout=20)
            if r.status_code == 200 and r.text.strip():
                return r
            status = r.status_code
        except requests.RequestException as e:
            status = type(e).__name__
        if attempt == len(BLOCK_WAITS):
            break
        wait = BLOCK_WAITS[attempt]
        with _NAVER_LOCK:
            until = time.time() + wait
            announce = until > _NAVER_COOLDOWN[0] + 5      # 같은 차단을 스레드마다 중복 출력하지 않기
            _NAVER_COOLDOWN[0] = max(_NAVER_COOLDOWN[0], until)
        if announce:
            print(f"\n  ⚠️ 네이버 응답 {status} (일시 차단) — 모든 검색을 {wait // 60}분 멈춘 뒤 재시도 "
                  f"({attempt + 1}/{len(BLOCK_WAITS)})")
    raise RuntimeError(f"네이버 응답 {status} — 약 {sum(BLOCK_WAITS) // 60}분을 기다려도 풀리지 않음")


def _paginate(session, first_html, max_pages, pbar=None, label="", stop_pages=None, first_next=None):
    """첫 화면에서 시작해 more API를 끝까지 따라가며 (링크, 관련뉴스 묶음, 페이지 수) 반환.
    stop_pages 가 있으면 그 쪽수에서 멈춘다(어차피 구간을 쪼갤 것이므로 더 넘기지 않음)."""
    links = _extract_links(first_html)
    related = _extract_related(first_html)
    if first_next is not None:                 # 바로 호출 방식: 첫 응답이 알려 준 다음 주소
        next_url = first_next
    else:
        m = _MORE_API_RE.search(first_html)
        next_url = _unescape_url(m.group(0)) if m else ""
    pages = 1
    while next_url and pages < max_pages and (stop_pages is None or pages < stop_pages):
        _polite_sleep()
        resp = _get_with_retry(session, next_url)
        try:
            data = resp.json()
        except ValueError:
            print(f"\n  ⚠️ JSON 파싱 실패 — {label} 여기서 중단")
            break
        coll = str(data.get("collection", ""))
        new = _extract_links(coll)
        if not new:
            break
        links.extend(new)
        related.update(_extract_related(coll))
        next_url = data.get("url") or ""
        pages += 1
        if pbar is not None:
            pbar.set_postfix({"구간": label, "p": pages, "링크": len(set(links))})
    return links, related, pages


def _search_first(session, query, news_office_id, ds, de):
    params = {
        "where": "news", "query": query, "sm": "tab_opt", "sort": 0,
        "pd": 3, "ds": f"{ds[:4]}.{ds[4:6]}.{ds[6:]}", "de": f"{de[:4]}.{de[4:6]}.{de[6:]}",
        "mynews": 1, "office_type": 1, "office_section_code": 1,
        "news_office_checked": news_office_id,
        "nso": f"so:r,p:from{ds}to{de},a:all",
    }
    return _get_with_retry(session, "https://search.naver.com/search.naver", params=params).text


def _search_direct_first(session, query, news_office_id, ds, de):
    """검색 첫 화면 없이 more API 를 start=1 로 바로 호출. (첫 응답 텍스트, 다음 주소) 반환. 실패하면 None."""
    params = {
        "query": query, "sort": 0, "pd": 3, "ds": f"{ds[:4]}.{ds[4:6]}.{ds[6:]}",
        "de": f"{de[:4]}.{de[4:6]}.{de[6:]}", "mynews": 1, "office_type": 1, "office_section_code": 1,
        "news_office_checked": news_office_id, "nso": f"so:r,p:from{ds}to{de},a:all",
        "start": 1, "sm": "tab_smr", "ssc": "tab.news.all", "photo": 0, "field": 0,
        "is_dts": 0, "rev": 0, "service_area": 0,
    }
    resp = _get_with_retry(session, _MORE_BASE, params=params)
    try:
        data = resp.json()
    except ValueError:
        return None
    if "collection" not in data:
        return None
    return str(data.get("collection", "")), (data.get("url") or "")


def _collect_adaptive(session, query, news_office_id, ds, de, max_pages, pbar, include_related, warns):
    """ds~de 구간 검색. 결과가 상한 근처면 반으로 나눠 다시 검색(재귀)."""
    label = f"{ds}~{de}" if ds != de else ds
    _polite_sleep()
    first = _search_direct_first(session, query, news_office_id, ds, de) if SEARCH_DIRECT else None
    splittable = de > ds
    if first is not None:
        html, first_next = first
        if not _extract_links(html):
            return []
        links, related, pages = _paginate(session, html, max_pages, pbar, label,
                                          stop_pages=SEARCH_SPLIT_PAGES if splittable else None,
                                          first_next=first_next)
    else:                                       # 기존 방식(첫 화면 → more API)
        html = _search_first(session, query, news_office_id, ds, de)
        if "api_noresult_wrap" in html and not _extract_links(html):
            return []
        links, related, pages = _paginate(session, html, max_pages, pbar, label,
                                          stop_pages=SEARCH_SPLIT_PAGES if splittable else None)

    if pages >= SEARCH_SPLIT_PAGES:
        d0, d1 = datetime.strptime(ds, "%Y%m%d"), datetime.strptime(de, "%Y%m%d")
        if splittable:
            mid = d0 + timedelta(days=(d1 - d0).days // 2)
            left = _collect_adaptive(session, query, news_office_id, ds, mid.strftime("%Y%m%d"),
                                     max_pages, pbar, include_related, warns)
            right = _collect_adaptive(session, query, news_office_id,
                                      (mid + timedelta(days=1)).strftime("%Y%m%d"), de,
                                      max_pages, pbar, include_related, warns)
            return links + left + right
        warns.append(f"{ds} 하루 결과가 {pages}쪽 이상 — 검색 상한에 걸려 일부 누락 가능(전량 수집 모드 권장)")

    if include_related:
        done = set()
        while related:
            docid, q = related.popitem()
            if docid in done:
                continue
            done.add(docid)
            _polite_sleep()
            rh = _get_with_retry(session, "https://search.naver.com/search.naver" + q
                                 + ("" if "where=" in q else "&where=news")).text
            rl, rr, _ = _paginate(session, rh, max_pages, None, label)
            links.extend(rl)
            for k, v in rr.items():
                if k not in done:
                    related.setdefault(k, v)
    return links


def collect_naver_news_links(query, news_office_id, start_date, end_date,
                             max_pages=10000, window_days=None, delay=None,
                             include_related=True, progress_path=None):
    """기간 내 검색 결과 링크 수집. 받지 못한 구간이 남으면 LinkCollectionIncomplete 를 올린다."""
    global SEARCH_DELAY
    if delay is not None:
        SEARCH_DELAY = delay
    window_days = window_days or SEARCH_WINDOW_DAYS
    session = requests.Session()
    session.headers.update({"User-Agent": _NAVER_UA, "Referer": "https://search.naver.com/",
                            "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7"})
    d0 = datetime.strptime(start_date, "%Y%m%d")
    d1 = datetime.strptime(end_date, "%Y%m%d")
    windows, cur = [], d0
    while cur <= d1:
        w_end = min(cur + timedelta(days=window_days - 1), d1)
        windows.append((cur.strftime("%Y%m%d"), w_end.strftime("%Y%m%d")))
        cur = w_end + timedelta(days=1)

    # 이어하기: 이미 끝난 구간은 저장된 링크를 그대로 쓴다
    done = {}
    if progress_path and os.path.isfile(progress_path):
        try:
            with open(progress_path, encoding="utf-8") as f:
                prev = json.load(f)
            if prev.get("query") == query and prev.get("window_days") == window_days:
                done = prev.get("done", {})
        except Exception:
            done = {}

    def _save():
        if progress_path:
            tmp = progress_path + ".tmp"
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump({"query": query, "window_days": window_days, "done": done}, f, ensure_ascii=False)
            os.replace(tmp, progress_path)

    todo = [w for w in windows if f"{w[0]}-{w[1]}" not in done]
    if len(todo) < len(windows):
        print(f"  (이어하기: {len(windows) - len(todo)}/{len(windows)}개 구간은 이미 받음)")
    failed, warns = [], []
    with tqdm(total=len(windows), initial=len(windows) - len(todo), desc="링크 수집") as pbar:
        for ds, de in todo:
            try:
                got = _collect_adaptive(session, query, news_office_id, ds, de,
                                        max_pages, pbar, include_related, warns)
                done[f"{ds}-{de}"] = list(dict.fromkeys(got))
                _save()
            except Exception as e:
                failed.append((ds, de))
                print(f"\n  ❌ 구간 {ds}~{de} 수집 실패: {e}")
            pbar.update(1)
            pbar.set_postfix({"링크": len({u for v in done.values() for u in v})})

    # 실패 구간은 한 번 더 (차단이 풀렸을 수 있으므로)
    if failed:
        print(f"\n  ↻ 실패한 {len(failed)}개 구간 다시 시도")
        retry, failed = failed, []
        for ds, de in retry:
            try:
                got = _collect_adaptive(session, query, news_office_id, ds, de,
                                        max_pages, None, include_related, warns)
                done[f"{ds}-{de}"] = list(dict.fromkeys(got))
                _save()
            except Exception as e:
                failed.append((ds, de))

    for w in warns:
        print(f"  ⚠️ {w}")
    links = list(dict.fromkeys(u for ds, de in windows for u in done.get(f"{ds}-{de}", [])))
    print(f"총 {len(links)}개의 뉴스 링크를 수집했습니다. (기간 {start_date}~{end_date})")
    if failed:
        raise LinkCollectionIncomplete(links, failed)
    return links


# 이미지 URL 수집 함수
def extract_image_urls(soup, collect_image=True, image_quality='medium'):
    """
    BeautifulSoup 객체에서 이미지 URL들을 추출하는 함수 (Yandex 방식 적용)

    ★ 핵심: 기사 본문(article#dic_area) 내부의 이미지만 추출 ★

    Args:
        soup: BeautifulSoup 파싱 객체
        collect_image: 이미지 수집 여부
        image_quality: 'high', 'medium', 'low'
    Returns:
        list: 이미지 URL 리스트
    """
    if not collect_image:
        return []

    image_urls = []

    try:
        # ★ 핵심: 기사 본문 영역만 선택 ★
        article_content = soup.select_one("article#dic_area")

        if not article_content:
            # article#dic_area가 없으면 다른 본문 영역 시도
            article_content = soup.select_one("article")

        if not article_content:
            # 본문 영역을 찾을 수 없으면 빈 리스트 반환
            return []

        # 1단계: 기사 본문 내부의 id가 img + 숫자인 이미지들
        # img1, img2, img3 등만 선택 (img_banner, img_logo 같은 건 제외)
        all_imgs_in_article = article_content.select("img")

        for img in all_imgs_in_article:
            img_id = img.get('id', '')
            # id가 'img'로 시작하고 그 뒤가 숫자인지 확인
            if img_id and img_id.startswith('img') and len(img_id) > 3 and img_id[3:].isdigit():
                # srcset 속성 우선 확인 (Yandex 방식)
                srcset = img.get('srcset', '')
                if srcset:
                    # srcset에서 첫 번째 이미지 URL 추출
                    first_url = srcset.split(',')[0].strip().split(' ')[0]
                    if 'imgnews.pstatic.net' in first_url:
                        image_urls.append(first_url)
                else:
                    # src 속성 사용
                    src = img.get('src', '')
                    if src and 'imgnews.pstatic.net' in src:
                        image_urls.append(src)

        # 2단계: id는 없지만 _LAZY_LOADING 클래스를 가진 이미지들 (본문 내부만)
        if not image_urls:
            lazy_imgs = article_content.select("img[class*='_LAZY_LOADING']")
            for img in lazy_imgs:
                # srcset 우선 확인
                srcset = img.get('srcset', '')
                if srcset:
                    first_url = srcset.split(',')[0].strip().split(' ')[0]
                    if 'imgnews.pstatic.net' in first_url:
                        image_urls.append(first_url)
                else:
                    src = img.get('src', '') or img.get('data-src', '')
                    if src and 'imgnews.pstatic.net' in src:
                        image_urls.append(src)

        # 3단계: 본문 내부의 모든 네이버 뉴스 이미지 (최후 백업)
        if not image_urls:
            for img in all_imgs_in_article:
                srcset = img.get('srcset', '')
                if srcset:
                    first_url = srcset.split(',')[0].strip().split(' ')[0]
                    if 'imgnews.pstatic.net' in first_url:
                        image_urls.append(first_url)
                else:
                    src = img.get('src', '')
                    if src and 'imgnews.pstatic.net' in src:
                        image_urls.append(src)

        # 중복 제거 및 빈 값 필터링 (순서 유지)
        image_urls = list(dict.fromkeys(image_urls))  # 순서 유지하면서 중복 제거
        image_urls = [url for url in image_urls if url and url.strip()]

        # 화질 조정
        image_urls = [adjust_image_quality(url, image_quality) for url in image_urls]

    except Exception as e:
        print(f"이미지 URL 추출 중 오류: {e}")

    return image_urls


# ── 문자열 정리 헬퍼 (두 번째 코드의 구체화된 변인 추출에 필요) ─────────────────
def _clean_text(s):
    """공백류 문자 정리 (줄바꿈은 보존)"""
    return re.sub(r"[ \t\xa0\u200b]+", " ", str(s or "")).strip()


def _flat_text(s):
    """줄바꿈까지 모두 한 줄 공백으로 정리 (2차 첨부 코드의 flat() 과 동일)"""
    return re.sub(r"\s+", " ", str(s or "")).strip()


# ── 장르(기사 유형) 추정 — 2번째 코드(NaverNewsCommentconditioncrawler)의 변인 구체화 반영 ──
GENRE_NAME = {1: "스트레이트", 2: "해설/분석", 3: "인터뷰", 4: "사설", 5: "칼럼/기고", 6: "기타"}
COLUMN_MARKS = ["칼럼", "기고", "시론", "발언대", "특별기고", "태평로", "동서남북", "에스프레소",
                "아침햇발", "세상읽기", "유레카", "만물상", "서초포럼", "뉴스룸에서", "현장에서",
                "朝鮮칼럼", "편집국에서", "슬기로운 기자생활"]
ANALYSIS_MARKS = ["뷰리핑", "뉴스 다이브", "논썰", "뉴스 저격", "분석", "팩트체크", "해설",
                  "이슈 인사이드", "심층", "흑백여의도"]
INTERVIEW_MARKS = ["인터뷰", "일문일답", "대담"]


def detect_genre(title, section, author):
    br = " ".join(re.findall(r"[\[\(【]([^\]\)】]{1,16})[\]\)】]", title or ""))
    title = title or ""
    section = section or ""
    author = author or ""
    if "사설" in br:
        return 4, "제목[사설]"
    if any(k in br for k in COLUMN_MARKS) or "칼럼" in title:
        return 5, "제목 칼럼마커"
    if any(k in author for k in ["논설위원", "논설실장", "주필", "편집인", "에디터"]):
        return 5, "바이라인 논설"
    if author and "기자" not in author and "특파원" not in author and len(author) > 2:
        return 5, "바이라인 외부필자"
    if any(k in br for k in INTERVIEW_MARKS) or "인터뷰" in title:
        return 3, "제목 인터뷰"
    if section == "오피니언":
        return 5, "섹션 오피니언"
    if any(k in br for k in ANALYSIS_MARKS):
        return 2, "제목 분석마커"
    return 1, "기본값"


# 기사 데이터 수집 함수
# ── 기존 크롤러의 기사 수집(요청·파싱) 로직은 그대로 두고, 수집 변인만
#    두 번째 코드(NaverNewsCommentconditioncrawler)의 구체화된 정의를 반영해 확장한다. ──
def _collect_article_details_html(media_name, url, collect_image=True, image_quality='medium'):
    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.content, "html.parser")

        # 제목 추출 (기존 방식 유지)
        title = ""

        # 1. 가장 정확한 제목 선택자로 시도
        title_span = soup.select_one("h2#title_area > span")
        if title_span:
            title = title_span.text.strip()

        # 2. 클래스로 찾기
        if not title:
            title_span = soup.select_one("h2.media_end_head_headline > span")
            if title_span:
                title = title_span.text.strip()

        # 3. 제목 영역 div에서 찾기
        if not title:
            title_div = soup.select_one("div.media_end_head_title")
            if title_div:
                spans = title_div.select("span")
                for span in spans:
                    span_text = span.text.strip()
                    if len(span_text) > 10:  # 제목으로 의미있는 길이
                        title = span_text
                        break

        # 4. 이전 방식으로 시도 (폴백)
        if not title:
            for h_tag in ['h1', 'h2', 'h3']:
                headers = soup.select(h_tag)
                for header in headers:
                    header_text = header.text.strip()
                    if header_text and len(header_text) > 10:
                        title = header_text
                        break
                if title:
                    break

        # 5. 마지막 방법: 긴 span 태그 찾기 (하지만 "구독해주세요"와 같은 불필요한 텍스트는 필터링)
        if not title:
            avoid_words = ["구독", "좋아요", "팔로우", "뉴스레터", "공유", "신청", "클릭"]
            for span in soup.select("span"):
                span_text = span.text.strip()
                # 광고나 구독 문구가 아닌 실제 제목만 추출
                if (span_text and len(span_text) > 15 and
                    not any(word in span_text for word in avoid_words)):
                    title = span_text
                    break

        # 작성자 추출
        writer = ""
        writer_element = soup.select_one("span.byline_s")
        if writer_element:
            writer = writer_element.text.strip()

        # ── 날짜/시각 추출 (구체화) ──────────────────────────────────────────
        # 2번째 코드처럼 표시 문자열이 아니라 data-date-time 속성(원자료)을 기준으로 삼는다.
        date = ""
        time_only = ""
        datetime_iso = ""
        date_el = soup.select_one("span._ARTICLE_DATE_TIME")
        raw_date = date_el.get("data-date-time", "").strip() if date_el else ""
        if raw_date:
            date = raw_date[:10]
            time_only = raw_date[11:16] if len(raw_date) >= 16 else ""
            if len(raw_date) >= 19:
                datetime_iso = f"{raw_date[:10]}T{raw_date[11:19]}+0900"
        else:
            # 폴백: 기존(첫 번째 코드) 방식 — 화면 표시 문자열 탐색
            date_elements = soup.select("span[class*='ARTICLE_DATE_TIME']")
            if date_elements:
                date = date_elements[0].text.strip()
            if not date:
                for span in soup.select("span"):
                    span_text = span.text.strip()
                    if ("202" in span_text) and ("." in span_text or ":" in span_text):
                        if re.search(r'\d{4}[./-]\d{1,2}[./-]\d{1,2}', span_text):
                            date = span_text
                            break

        # 수정 시각 (원본 속성값 + ISO 결합본 둘 다 보존)
        modify_date = ""
        modify_datetime = ""
        modify_el = soup.select_one("span._ARTICLE_MODIFY_DATE_TIME")
        if modify_el:
            raw_modify = modify_el.get("data-modify-date-time", "").strip()
            modify_date = raw_modify
            if raw_modify and len(raw_modify) >= 19:
                modify_datetime = f"{raw_modify[:10]}T{raw_modify[11:19]}+0900"

        # 섹션 정보 추출
        section = ""
        # 1. 'media_end_categorize_item' 클래스로 시도
        section_element = soup.select_one("em.media_end_categorize_item")
        if section_element:
            section = section_element.text.strip()

        # 2. 다른 선택자 시도 (백업)
        if not section:
            # 다른 가능한 섹션 선택자들 시도
            section_selectors = [
                "div.media_end_categorize_item",
                "a.categorize_item",
                "div.article_category",
                "span.category"
            ]

            for selector in section_selectors:
                section_elem = soup.select_one(selector)
                if section_elem and section_elem.text.strip():
                    section = section_elem.text.strip()
                    break

        # 3. 마지막 백업: 메타 태그에서 찾기
        if not section:
            meta_keywords = soup.select_one("meta[property='article:section'], meta[name='article:section']")
            if meta_keywords and meta_keywords.get('content'):
                section = meta_keywords.get('content')

        # 이미지 URL 추출 (기존 방식 유지)
        image_urls = extract_image_urls(soup, collect_image, image_quality)

        # ── 신규 수집 변수 (기존 크롤러에 이미 있던 것들) ───────────────────

        # PICK 여부 (언론사 자체 강조 기사)
        pick = 1 if soup.select_one("i.media_end_head_channel_pick") else 0

        # 면 정보 (예: '4면 1단', 'A1면 3단' — 언론사마다 형식 상이 → raw 저장)
        page = ""
        for span in soup.select("span.sds-comps-text-weight-sm"):
            span_text = span.text.strip()
            if "면" in span_text:
                page = span_text
                break

        # 전체 섹션 목록 (다중 섹션 분류 지원)
        section_items = [el.text.strip() for el in soup.select("em.media_end_categorize_item") if el.text.strip()]
        sections = "|".join(section_items)
        sec_count = len(section_items)

        # 반응("이 기사를 추천합니다") 수집 — 총합 + 5종 세분화
        #   쏠쏠정보(useful) · 흥미진진(wow) · 공감백배(touched) · 분석탁월(analytical) · 후속강추(recommend)
        # ★ ul._faceLayer / li.u_likeit_list 같은 감싸는 클래스에 의존하지 않고,
        #   data-type 속성을 가진 <a> 태그를 문서 전체에서 직접 찾는다.
        #   (감싸는 wrapper의 클래스명은 페이지 버전에 따라 바뀔 수 있지만,
        #    data-type="useful/wow/touched/analytical/recommend" 값 자체는 안정적이다.)
        react_total = 0
        react_useful = react_wow = react_touched = react_analytic = react_recommend = 0
        react_total_el = soup.select_one("span.u_likeit_text._count")
        if react_total_el:
            try: react_total = int(react_total_el.text.strip().replace(",", ""))
            except: pass

        REACT_TYPE_MAP = {
            "useful": "useful", "wow": "wow", "touched": "touched",
            "analytical": "analytical", "recommend": "recommend",
        }
        for btn in soup.select("a[data-type]"):
            dtype = btn.get("data-type", "")
            if dtype not in REACT_TYPE_MAP:
                continue  # 감정표현(good/warm/sad/angry/want) 등 다른 위젯의 data-type은 제외
            cnt = btn.select_one("span._count") or btn.select_one("span.u_likeit_list_count")
            if not cnt:
                continue
            try: val = int(cnt.text.strip().replace(",", ""))
            except: val = 0
            if dtype == "useful":       react_useful    = val
            elif dtype == "wow":        react_wow       = val
            elif dtype == "touched":    react_touched   = val
            elif dtype == "analytical": react_analytic  = val
            elif dtype == "recommend":  react_recommend = val

        # AI 요약 제공 여부

        # 본문 내 동영상 포함 여부
        article_body = soup.select_one("article#dic_area")
        has_video = 0
        if article_body:
            has_video = 1 if article_body.select_one("div._VIDEO_AREA, video, iframe[src*='video']") else 0

        # 언론사 주요뉴스 아웃링크 수 (자기참조 밀도)
        outlink_n = len(soup.select("ul.media_end_linked_list > li"))

        # 연결 이슈 수 및 이슈명 (의제 프레이밍 네트워크)
        issue_items = [el.text.strip() for el in soup.select("strong.related_issue_subject_name") if el.text.strip()]
        issue_n = len(issue_items)
        issues  = "|".join(issue_items)

        # 기자 네이버 ID
        journalist_id = ""
        jid_el = soup.select_one("div._JOURNALIST_CARD div._JOURNALIST_ID")
        if jid_el:
            journalist_id = jid_el.get("data-journalistid", "").strip()

        # 원문 링크 존재 여부 (플랫폼 종속성)
        has_origin = 1 if soup.select_one("a.media_end_head_origin_link") else 0

        # TTS 서비스 제공 여부 (포맷 다양화)
        has_tts = 1 if soup.select_one("div.media_end_head_tts") else 0

        # 기사 고유 ID (URL에서 파싱 — 패널·종단 분석용 키)
        article_id = ""
        import re as _re
        aid_match = _re.search(r'/article/\d+/(\d+)', url)
        if aid_match:
            article_id = aid_match.group(1)

        # 언론사 oid / 기사 aid 분리 저장 (2번째 코드의 naver_oid/naver_aid 구체화 반영)
        naver_oid, naver_aid = "", ""
        oid_aid_match = _re.search(r"/article/(?:comment/)?(\d{3,4})/(\d{7,11})", url)
        if oid_aid_match:
            naver_oid, naver_aid = oid_aid_match.group(1), oid_aid_match.group(2)

        # 언론사 유형 (JS 변수에서 파싱: '방송/통신사', '종합일간지' 등)
        office_type = ""
        raw_html = response.text
        otype_match = _re.search(r'office\s*=\s*\{[^}]*category\s*:\s*"([^"]+)"', raw_html)
        if otype_match:
            office_type = otype_match.group(1).strip()

        # og:description (네이버 생성 요약 — 본문과 의미 괴리 분석용)
        og_desc = ""
        og_el = soup.select_one("meta[property='og:description']")
        if og_el:
            og_desc = og_el.get("content", "").strip()

        # 댓글 허용 여부 (cbox JS 내 차단 문구 탐지)
        comment_allow = 0 if 'info_comment_off' in raw_html else 1

        # 네이버 섹션 코드 (URL sid 파라미터 — 언론사 분류와 교차 검증용)
        naver_sid = ""
        sid_match = _re.search(r'[?&]sid=(\d+)', url)
        if sid_match:
            naver_sid = sid_match.group(1)

        # 기사 포맷 유형 (JS article.type: 0=텍스트, 1=포토, 2=TV)
        article_type = ""
        atype_match = _re.search(r'article\s*=\s*\{[^}]*type:\s*"(\d)"', raw_html)
        if atype_match:
            article_type = atype_match.group(1)

        # 대표 썸네일 URL (og:image)
        thumb_url = ""
        og_img_el = soup.select_one("meta[property='og:image']")
        if og_img_el:
            thumb_url = og_img_el.get("content", "").strip()

        # 댓글 개수 (id="comment_count" / class="_COMMENT_COUNT_VIEW")
        # ★ 실측 결과, 정적 HTML(requests) 안의 이 span 은 항상 "댓글"이라는
        #   자리표시자 문자열만 담고 있고 실제 숫자는 JS 로 채워진다(브라우저에서만 보임).
        #   그래서 span 텍스트를 그대로 파싱하면 항상 0으로 남는다.
        #   실제 숫자는 댓글 위젯이 내부적으로 호출하는 cbox 카운트 API 응답에만 있어서,
        #   여기서 같은 API를 가벼운 1페이지 요청으로 한 번 더 불러 그 값을 가져온다.
        comment_n = 0      # 댓글 수·집계·작성자 통계는 collect_article_details 에서 댓글 API로 채운다

        # 네이버 전역 기사 ID (gdid — 고유 식별 보완 키)
        gdid = ""
        gdid_match = _re.search(r'gdid["\s]*:["\s]*"([^"]+)"', raw_html)
        if not gdid_match:
            gdid_meta = soup.select_one("meta[name='napp-site-analysis']")
            if gdid_meta:
                g2 = _re.search(r'gdid=([^,]+)', gdid_meta.get("content", ""))
                if g2:
                    gdid = g2.group(1).strip()
        else:
            gdid = gdid_match.group(1).strip()

        # ── 본문 분해: 부제·사진설명·중간제목을 본문에서 분리 ──────────────────
        # ★ 2번째 코드(NaverNewsCommentconditioncrawler)의 핵심 구체화 지점.
        #   네이버는 부제(strong.media_end_summary)와 사진설명(span.end_photo_org)을
        #   article#dic_area 안에 그대로 넣어 두므로, 분리하지 않으면 본문(text)이 오염된다.
        subtitle = captions = subheads = fulltext = summary_box = ""
        img_n = 0
        art = soup.select_one("article#dic_area")
        if art:
            # 원본 soup(이미지 추출 등에 재사용됨)을 건드리지 않도록 복제본에서 작업
            a = BeautifulSoup(str(art), "html.parser").select_one("article")
            img_n = len(a.select("img"))

            # 언론사가 본문 첫머리에 넣은 요약 상자(예: 조선일보 '3줄 요약')는 본문과 따로 저장
            _boxes = a.select(".navernews_end_summary")
            summary_box = " || ".join(_clean_text(e.get_text(" ", strip=True)) for e in _boxes)
            for e in _boxes:
                e.decompose()

            subs = []
            # 1. 클래스가 명시된 공식 부제 추출
            for e in a.select("strong.media_end_summary, div.media_end_summary, h3.media_end_summary"):
                subs.append(_clean_text(e.get_text(" ", strip=True))); e.decompose()

            # 2. class가 없는 가짜 부제(본문 최상단의 b, strong 태그) 추출 및 제거
            for e in a.find_all(['b', 'strong']):
                txt = _clean_text(e.get_text(" ", strip=True))
                if not txt or len(txt) < 5:
                    continue  # 너무 짧은 단어는 무시

                # (A) 본문 시작 부근인지 확인 (이 태그 앞의 텍스트가 40자 이내인지 계산)
                prev_len = 0
                for el in a.descendants:
                    if el == e:
                        break
                    if isinstance(el, str):
                        prev_len += len(el.strip())

                if prev_len > 40:
                    continue  # 기사 중간에 등장하는 강조 텍스트면 패스

                # (B) 인라인 여부 확인 (본문의 일부인지, 독립된 부제인지)
                is_inline = False
                for el in e.next_elements:
                    if el in e.descendants:
                        continue  # 자기 자신 내부의 태그는 건너뜀

                    if isinstance(el, str):
                        if el.strip():
                            is_inline = True  # 바로 텍스트가 이어지면 본문의 일부로 간주
                            break
                    elif getattr(el, 'name', None) in ['br', 'p', 'div', 'article']:
                        break  # 텍스트가 나오기 전에 줄바꿈(<br>)을 만나면 독립된 부제가 맞음

                # 조건에 부합하면 부제로 편입하고 본문에서 삭제
                if not is_inline:
                    subs.append(txt)
                    e.decompose()

            subtitle = " | ".join(x for x in subs if x)

            caps = []
            for e in a.select("span.end_photo_org, em.img_desc, div.nbd_table"):
                c = _clean_text(e.get_text(" ", strip=True))
                if c: caps.append(c)
                e.decompose()
            captions = " || ".join(dict.fromkeys(caps))

            heads = [_clean_text(e.get_text(" ", strip=True)) for e in a.select("strong, b, h3, h4")]
            subheads = " || ".join(dict.fromkeys(h for h in heads if h and len(h) < 120))

            for e in a.select("script, style"):
                e.decompose()
            for br in a.select("br"):
                br.replace_with("\n")
            raw_txt = re.sub(r"[ \t\xa0\u200b]+", " ", a.get_text(""))
            raw_txt = "\n".join(l.strip() for l in raw_txt.split("\n"))
            fulltext = re.sub(r"\n{3,}", "\n\n", raw_txt).strip()

        # text / text_len 은 부제·사진설명이 걷힌 fulltext 기준으로 재계산한다
        # (기존 코드의 text는 본문 전체를 그대로 이어붙여 부제·사진설명이 섞여 있었다)
        text = _flat_text(fulltext)
        text_len = len(fulltext)
        fulltext_available = 1 if text_len >= 50 else 0

        # 장르(기사 유형) 추정
        genre, genre_rule = detect_genre(title, section, writer)
        genre_label = GENRE_NAME.get(genre, "")

        return {
            "media":            media_name,
            "office_type":      office_type,
            "article_id":       article_id,
            "gdid":             gdid,
            "title":            title,
            "subtitle":         subtitle,
            "url":              url,
            "naver_sid":        naver_sid,
            "naver_oid":        naver_oid,
            "naver_aid":        naver_aid,
            "date":             date,
            "time":             time_only,
            "datetime":         datetime_iso,
            "modify_date":      modify_date,
            "modify_datetime":  modify_datetime,
            "article_type":     article_type,
            "genre":            genre,
            "genre_label":      genre_label,
            "genre_rule":       genre_rule,
            "fulltext":         fulltext,
            "text":             text,
            "text_len":         text_len,
            "fulltext_available": fulltext_available,
            "captions":         captions,
            "subheads":         subheads,
            "og_desc":          og_desc,
            "thumb_url":        thumb_url,
            "writer":           writer,
            "journalist_id":    journalist_id,
            "section":          section,
            "sections":         sections,
            "sec_count":        sec_count,
            "page":             page,
            "pick":             pick,
            "react_total":      react_total,
            "react_useful":     react_useful,
            "react_wow":        react_wow,
            "react_touched":    react_touched,
            "react_analytic":   react_analytic,
            "react_recommend":  react_recommend,
            "summary_box":      summary_box,
            "has_summary_box":  1 if summary_box else 0,
            "has_video":        has_video,
            "has_tts":          has_tts,
            "has_origin":       has_origin,
            "img_n":            img_n,
            "comment_n":        comment_n,
            "comment_allow":    comment_allow,
            "outlink_n":        outlink_n,
            "issue_n":          issue_n,
            "issues":           issues,
            "images":           "|".join(image_urls)
        }
    except Exception as e:
        # 오류가 발생해도 조용히 빈 데이터 반환
        return {
            "media":            media_name,
            "office_type":      "",
            "article_id":       "",
            "gdid":             "",
            "title":            "",
            "subtitle":         "",
            "url":              url,
            "naver_sid":        "",
            "naver_oid":        "",
            "naver_aid":        "",
            "date":             "",
            "time":             "",
            "datetime":         "",
            "modify_date":      "",
            "modify_datetime":  "",
            "article_type":     "",
            "genre":            "",
            "genre_label":      "",
            "genre_rule":       "",
            "fulltext":         "",
            "text":             "",
            "text_len":         0,
            "fulltext_available": 0,
            "captions":         "",
            "subheads":         "",
            "og_desc":          "",
            "thumb_url":        "",
            "writer":           "",
            "journalist_id":    "",
            "section":          "",
            "sections":         "",
            "sec_count":        0,
            "page":             "",
            "pick":             0,
            "react_total":      0,
            "react_useful":     0,
            "react_wow":        0,
            "react_touched":    0,
            "react_analytic":   0,
            "react_recommend":  0,
            "summary_box":      "",
            "has_summary_box":  0,
            "has_video":        0,
            "has_tts":          0,
            "has_origin":       0,
            "img_n":            0,
            "comment_n":        0,
            "comment_allow":    0,
            "outlink_n":        0,
            "issue_n":          0,
            "issues":           "",
            "images":           ""
        }

# ── 엑셀용 .xlsx 사본 저장 ─────────────────────────────────────────────────────
# CSV 는 "이 칸은 글자"라는 정보를 담을 수 없어서, 엑셀로 더블클릭해 열면
#   · mark "-ksn1…" / 댓글 "-_-", "=ㅋㅋ"  → 수식으로 오인 → #NAME?
#   · commentNo 18자리                    → 15자리에서 잘림(뒷자리 000)
#   · naver_oid "001", naver_aid "0016…"  → 앞의 0 이 사라짐
# 같은 문제가 생긴다. 그래서 CSV(분석용 원본) 옆에 같은 이름의 .xlsx 를 함께
# 저장한다. xlsx 는 칸마다 '글자' 로 저장되므로 더블클릭해도 값이 그대로 보인다.
XLSX_TEXT_COLS = ["commentNo", "mark", "parentCommentNo", "article_id", "gdid",
                  "naver_oid", "naver_aid", "naver_sid", "journalist_id"]
EXCEL_MAX_ROWS = 1_048_575
_XLSX_WARNED = False

def _write_excel_copy(csv_path):
    global _XLSX_WARNED
    try:
        import xlsxwriter  # noqa: F401
    except ImportError:
        if not _XLSX_WARNED:
            print("  ⚠️ .xlsx 사본을 만들려면 'pip install xlsxwriter' 후 커널을 재시작하세요 (CSV 는 정상 저장됨)")
            _XLSX_WARNED = True
        return
    try:
        df = pd.read_csv(csv_path, dtype={c: str for c in XLSX_TEXT_COLS}, keep_default_na=False)
        if len(df) > EXCEL_MAX_ROWS:
            print(f"  ⚠️ {len(df):,}행 — 엑셀 한도(1,048,575행) 초과라 .xlsx 사본 생략: {csv_path}")
            return
        xlsx_path = csv_path[:-4] + ".xlsx"
        tmp_path = xlsx_path[:-5] + ".tmp.xlsx"
        with pd.ExcelWriter(tmp_path, engine="xlsxwriter",
                            engine_kwargs={"options": {"strings_to_formulas": False,   # "=…", "-…" 를 글자로
                                                       "strings_to_numbers": False,    # "001" 을 글자로
                                                       "strings_to_urls": False}}) as w:
            df.to_excel(w, index=False, sheet_name="data")
        os.replace(tmp_path, xlsx_path)
    except Exception as e:
        print(f"  ⚠️ .xlsx 사본 저장 실패({csv_path}): {e} — CSV 는 정상")

# ── 병렬 댓글 수집 ───────────────────────────────────────────────────────────
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed as _as_completed

_thread_local = threading.local()

def _get_thread_session(referer_url):
    if not hasattr(_thread_local, "session"):
        s = requests.Session()
        s.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36",
            "Accept": "application/json, text/javascript, */*; q=0.01",
            "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8",
            "X-Requested-With": "XMLHttpRequest",
        })
        _thread_local.session = s
    _thread_local.session.headers["Referer"] = referer_url
    return _thread_local.session


# ── 댓글 전수 수집 로직 (2번째 코드의 fetch_comments 를 반영) ─────────────────
# 기존(1번 코드) 로직과 달리:
#   1) includeAllStatus=true 를 붙여 삭제·클린봇 처리된 댓글까지 모두 받는다
#      (이게 "전수(全數)" 수집의 핵심 — 화면에서 사라진 댓글도 카운트에서 빠지지 않는다)
#   2) page 파라미터만 증가시키는 것이 아니라 cbox 응답의 morePage.next 값을
#      moreParam.next 로 되돌려 보내는 방식으로 페이지네이션한다.
#      (cbox 는 page 파라미터만으로는 종종 페이지를 건너뛰거나 중복시킨다)
#   3) 신규 댓글이 0건인 페이지가 나와도 즉시 멈추지 않고, 연속 stall 허용치
#      (COMMENT_STALL_MAX)까지는 계속 다음 페이지를 시도한다.
CBOX_URL = "https://apis.naver.com/commentBox/cbox/web_naver_list_jsonp.json"
COMMENT_SORT = "old"        # old(등록순) / new / favorite
COMMENT_PAGE_MAX = 1000
COMMENT_STALL_MAX = 12      # 신규 0인 페이지를 몇 번까지 참을지 (cbox 페이지 불규칙성 대응)
COMMENT_GAP = 0.15

# 댓글에서 수집할 필드 (전수 수집이므로 삭제·클린봇 판정에 필요한 필드까지 포함)
# ★ mark: userIdNo 가 최근 네이버 쪽에서 항상 빈 값("")으로 내려오게 되면서
#   (실측 확인 — 프라이버시 정책 변경으로 추정) 대신 쓸 수 있는 필드.
#   댓글 위젯의 "댓글모음"(이 사용자의 다른 댓글 보기) 버튼이 내부적으로 쓰는
#   data-param 의 targetMark 값과 100% 동일함을 실제 기사로 검증했다 —
#   즉 네이버가 동일인 여부 조회에 실제로 쓰는 값이므로 userIdNo 의 사실상
#   대체재로 볼 수 있다. userName/maskedUserId 는 앞 4글자만 보이는 마스킹이라
#   서로 다른 사람이 같은 값으로 겹칠 수 있으니(실측으로 확인됨), 동일인 판별은
#   userName 이 아니라 이 mark 값 기준으로 하는 것이 더 정확하다.
# ★ userIdNo 컬럼 삭제 (2026.09 실측: 100건 중 0건 채워짐 — 항상 빈 값). 동일인 판별은 mark 로.
# ★ defamation · blind 추가: 삭제 사유 문구를 네이버와 똑같이 판정하는 데 필요한 값
# ★ parentCommentNo·replyLevel·replyAllCount 추가: 답글(replyLevel=2)도 전수 수집하므로 원댓글과 구분·연결용
COMMENT_FIELDS = ["commentNo", "parentCommentNo", "replyLevel", "replyAllCount",
                   "mark", "userName", "contents", "regTime", "modTime",
                   "sympathyCount", "antipathyCount", "deleted", "hiddenByCleanbot",
                   "status", "defamation", "blind", "inspectionId"]

# ── 댓글 조치유형 분류 · 생존시간 분석 ─────────────────────────────────────────
# ★ 2026.09 수정: 네이버 댓글 위젯(cbox.js)의 표시 로직을 그대로 옮겼다.
#   네이버가 삭제 댓글 문구를 고르는 함수(minified 'de')는 다음과 같다.
#     status 1                → "작성자에 의해 삭제된 댓글입니다."
#     status 3 & !defamation  → "운영규정 미준수로 인해 삭제된 댓글입니다."
#     status 3 & defamation   → "정보통신망법에 따른 권리침해 요청이 있어, 게시중단 되었습니다."
#     status 11               → "콘텐츠 제공자에 의해 댓글이 삭제되었습니다."
#     그 외                   → "댓글이 삭제되었습니다." (defamation 이면 권리침해 문구)
#   삭제되지 않았지만 blind 인 댓글: status 11 → 콘텐츠 제공자 삭제, 15 → 권리침해, 그 외 임시 블라인드
#   (이전 코드는 inspectionId 유무로 권리침해를 판정했는데, 실측 결과 운영규정 미준수 삭제에도
#    inspectionId 가 붙어 있어 운영규정 미준수 삭제가 전부 권리침해로 잘못 분류됐다.)
ACTION_TEXT = {
    "작성자에 의해 삭제된 댓글입니다.":                             "작성자삭제",
    "운영규정 미준수로 인해 삭제된 댓글입니다.":                     "운영규정미준수삭제",
    "정보통신망법에 따른 권리침해 요청이 있어, 게시중단 되었습니다.": "권리침해게시중단",
    "콘텐츠 제공자에 의해 댓글이 삭제되었습니다.":                   "콘텐츠제공자삭제",
    "댓글이 삭제되었습니다.":                                       "삭제(사유미표시)",
    "신고로 인해 임시 블라인드 되었습니다.":                         "임시블라인드",
    "클린봇이 부적절한 표현을 감지한 댓글입니다.":                   "클린봇",
}
TEXT_OF = {v: k for k, v in ACTION_TEXT.items()}

# get_comments() 가 COMMENT_FIELDS 뒤에 이어붙이는 분석 파생 필드
COMMENT_DERIVED_FIELDS = ["action_type", "action_text", "action_source",
                           "action_time", "survival_min", "survival_day",
                           "comment_text", "cleanbot_original",
                           "is_deleted", "is_cleanbot"]


def judge_action(c):
    """조치유형 판정 — 네이버 댓글 위젯의 화면 문구 선택 로직과 동일한 순서·조건."""
    st = str(c.get("status"))
    defam = bool(c.get("defamation"))
    if c.get("deleted"):
        if st == "1":
            return "작성자삭제", "JSON_status1"
        if st == "3":
            return ("권리침해게시중단", "JSON_status3+defamation") if defam else ("운영규정미준수삭제", "JSON_status3")
        if st == "11":
            return "콘텐츠제공자삭제", "JSON_status11"
        return ("권리침해게시중단", f"JSON_status{st}+defamation") if defam else ("삭제(사유미표시)", f"JSON_status{st}")
    if c.get("blind"):
        if st == "11":
            return "콘텐츠제공자삭제", "JSON_blind_status11"
        if st == "15":
            return "권리침해게시중단", "JSON_blind_status15"
        return "임시블라인드", f"JSON_blind_status{st}"
    if c.get("hiddenByCleanbot"):
        return "클린봇", "JSON_cleanbot"
    return "노출", "JSON_정상"


def _augment_comment(c):
    """원본 댓글 dict(c) → COMMENT_DERIVED_FIELDS 순서의 분석값 리스트."""
    atype, src = judge_action(c)
    reg, mod = c.get("regTime", ""), c.get("modTime", "")
    acted = atype not in ("노출", "클린봇")          # 삭제·게시중단·블라인드 = 조치됨
    action_time = mod if acted else ""
    survival_min = survival_day = ""
    if acted and reg and mod:
        try:
            survival_min = round((pd.Timestamp(mod) - pd.Timestamp(reg)).total_seconds() / 60, 1)
            survival_day = round(survival_min / 1440, 2)
        except Exception:
            pass
    body = str(c.get("contents") or "")
    shown = TEXT_OF.get(atype, "")
    if str(c.get("replyLevel", "1")) != "1" and atype in ("작성자삭제", "운영규정미준수삭제"):
        shown = shown.replace("댓글입니다", "답글입니다")      # 네이버 화면 문구(답글용)와 동일하게
    # 조치된 댓글은 화면에 뜨는 대체문구를 comment_text 에 채운다
    comment_text = body if atype in ("노출", "클린봇") else shown
    # 클린봇은 화면에서만 가려질 뿐 원문이 API 응답에는 그대로 남아 있다
    cleanbot_original = body if atype == "클린봇" else ""
    return [atype, shown, src, action_time, survival_min, survival_day,
            comment_text, cleanbot_original,
            1 if c.get("deleted") else 0, 1 if c.get("hiddenByCleanbot") else 0]


_COMMENT_CHECK = {}                     # url → 네이버 집계 대비 수집 결과 (전수 여부 확인용)
_CHECK_LOCK = threading.Lock()


def _cbox_page(session, params, tries=4):
    """댓글 API 1회 호출(JSONP). 일시 오류·차단은 재시도, 끝내 실패하면 None."""
    for k in range(tries):
        try:
            resp = session.get(CBOX_URL, params=params, timeout=15)
            if resp.status_code == 200:
                mm = re.search(r"_callback\((.*)\);?\s*$", resp.text, re.S)
                body = mm.group(1) if mm else (re.search(r"\{.*\}", resp.text, re.S) or [None])[0]
                if body:
                    data = json.loads(body)
                    if data.get("success", True):
                        return data
            elif resp.status_code in (403, 429):
                time.sleep(30 * (k + 1))
                continue
        except Exception:
            pass
        time.sleep(2 * (k + 1))
    return None


def _cbox_list(session, object_id, parent=None):
    """원댓글 목록(parent=None) 또는 특정 댓글의 답글 목록을 끝까지 받는다.
    반환: (댓글 dict 목록, 첫 페이지 result, 성공 여부)"""
    got, seen, next_param, stall, first, ok = [], set(), "", 0, None, True
    for page in range(1, COMMENT_PAGE_MAX + 1):
        params = {
            "ticket": "news", "templateId": "default_society", "pool": "cbox5",
            "lang": "ko", "country": "KR", "objectId": object_id,
            "pageSize": 100, "indexSize": 10, "page": page, "sort": COMMENT_SORT,
            "includeAllStatus": "true",
        }
        if parent:
            params["parentCommentNo"] = parent
        if next_param:
            params["pageType"] = "more"
            params["moreParam.next"] = next_param
        data = _cbox_page(session, params)
        if data is None:
            ok = False
            break
        result = data.get("result") or {}
        if first is None:
            first = result
        clist = result.get("commentList") or []
        new = 0
        for c in clist:
            no = str(c.get("commentNo", ""))
            if parent and no == str(parent):
                continue                                   # 답글 목록에 섞여 오는 원댓글 자신은 제외
            if no and no not in seen:
                seen.add(no); got.append(c); new += 1
        next_param = (result.get("morePage") or {}).get("next", "")
        if not clist:
            break
        stall = stall + 1 if new == 0 else 0
        if stall >= COMMENT_STALL_MAX or not next_param:
            break
        time.sleep(COMMENT_GAP)
    return got, first, ok


def get_comments(story_url, include_replies=True):
    """기사 1건의 댓글 + 답글 전수(삭제·클린봇 포함). 네이버 집계와 비교한 결과를 _COMMENT_CHECK 에 남긴다."""
    m = re.search(r"article/(?:comment/)?(\d{3})/(\d{10})", story_url or "")
    if not m:
        return []
    object_id = f"news{m.group(1)},{m.group(2)}"
    session = _get_thread_session(story_url)
    tops, first, ok_top = _cbox_list(session, object_id)
    replies, ok_rep = [], True
    if include_replies:
        for c in tops:
            if (c.get("replyAllCount") or c.get("replyCount") or 0) > 0:
                r, _, ok = _cbox_list(session, object_id, parent=c.get("commentNo"))
                replies.extend(r)
                ok_rep = ok_rep and ok
    cnt = (first or {}).get("count") or {}
    exp_c, exp_r = int(cnt.get("comment") or 0), int(cnt.get("reply") or 0)
    with _CHECK_LOCK:
        _COMMENT_CHECK[story_url] = {
            "url": story_url, "naver_comment": exp_c, "naver_reply": exp_r,
            "got_comment": len(tops), "got_reply": len(replies),
            "request_ok": bool(ok_top and ok_rep and first is not None),
            "complete": bool(ok_top and ok_rep and first is not None
                             and len(tops) >= exp_c and (not include_replies or len(replies) >= exp_r)),
        }
    return [[c.get(f, "") for f in COMMENT_FIELDS] + _augment_comment(c) for c in tops + replies]


def _comment_worker(args):
    """기사 1건 댓글 수집. 요청 실패로 덜 받았으면 잠시 뒤 최대 2번 더 시도한다."""
    row, media_name = args
    url = row.get("url", "")
    if not isinstance(url, str) or not url:
        return []
    rows = []
    for attempt in range(3):
        try:
            rows = get_comments(url)
        except Exception as e:
            print(f"  워커 오류: {url}: {e}")
        chk = _COMMENT_CHECK.get(url, {})
        if chk.get("request_ok"):
            break
        time.sleep(20 * (attempt + 1))
    return [[url, media_name, row.get("title", "")] + c for c in rows]


def _write_comment_check(comments_dir, output_file_prefix, media_name, urls):
    """언론사별 '네이버 집계 대비 수집' 확인표를 저장하고 요약을 돌려준다."""
    import csv as _csv
    recs = [_COMMENT_CHECK[u] for u in urls if u in _COMMENT_CHECK]
    if not recs:
        return None
    df = pd.DataFrame(recs)
    path = os.path.join(comments_dir, f"check_{output_file_prefix}_{media_name}.csv")
    _atomic_write_csv(df, path, index=False, encoding="utf-8-sig", quoting=_csv.QUOTE_ALL)
    s = {"articles": len(df), "naver_comment": int(df.naver_comment.sum()), "naver_reply": int(df.naver_reply.sum()),
         "got_comment": int(df.got_comment.sum()), "got_reply": int(df.got_reply.sum()),
         "incomplete": int((~df.complete).sum())}
    with open(path[:-4] + ".json", "w", encoding="utf-8") as f:
        json.dump(s, f, ensure_ascii=False)
    return s


def write_run_report(base_path, queries, news_offices, start_date, end_date, use_query=True):
    """모든 키워드·언론사의 수집 결과를 한 장으로 정리: 수집보고서.txt / 수집보고서.csv"""
    import glob as _g, csv as _csv
    rows = []
    kws = [q.strip('"').strip() for q in queries] if use_query else ["전체기사"]
    for kw in kws:
        kw_dir = os.path.join(base_path, kw)
        for media_name in news_offices:
            art = os.path.join(kw_dir, media_name, "articles.csv")
            r = {"키워드": kw, "언론사": media_name, "기사": "", "댓글(수집/네이버)": "", "답글(수집/네이버)": "",
                 "댓글 덜 받은 기사": "", "상태": ""}
            if not os.path.isfile(art):
                prog = os.path.join(kw_dir, media_name, "_links_progress.json")
                r["상태"] = "❌ 미완료 — 다시 실행하면 이어서 수집" if os.path.isfile(prog) else "❌ 수집 안 됨"
                rows.append(r); continue
            try:
                r["기사"] = len(pd.read_csv(art, usecols=["url"]))
            except Exception:
                r["기사"] = "?"
            cj = os.path.join(kw_dir, "comments", f"check_{kw}_{media_name}.json")
            if os.path.isfile(cj):
                with open(cj, encoding="utf-8") as f:
                    s = json.load(f)
                r["댓글(수집/네이버)"] = f"{s['got_comment']:,}/{s['naver_comment']:,}"
                r["답글(수집/네이버)"] = f"{s['got_reply']:,}/{s['naver_reply']:,}"
                r["댓글 덜 받은 기사"] = s["incomplete"]
                r["상태"] = "✓ 완료" if s["incomplete"] == 0 else "⚠️ 일부 기사 댓글 부족(확인표 참고)"
            else:
                r["상태"] = "✓ 기사 완료 (댓글 미수집)"
            rows.append(r)
    df = pd.DataFrame(rows)
    _atomic_write_csv(df, os.path.join(base_path, "수집보고서.csv"), index=False, encoding="utf-8-sig",
                      quoting=_csv.QUOTE_ALL)
    done = (df["상태"].str.startswith("✓")).sum()
    lines = [f"네이버 뉴스 수집 보고서  ({time.strftime('%Y-%m-%d %H:%M')})",
             f"기간 {start_date} ~ {end_date} · 키워드 {', '.join(kws)} · 언론사 {len(news_offices)}곳",
             f"완료 {done}/{len(df)} (키워드×언론사)", "",
             df.to_string(index=False), "",
             "· 댓글(수집/네이버): 우리가 받은 수 / 네이버가 그 기사들에 대해 집계한 수. 같으면 전수 수집.",
             "· ❌ 미완료가 있으면 같은 설정으로 다시 실행하세요. 끝난 부분은 건너뛰고 이어서 받습니다."]
    with open(os.path.join(base_path, "수집보고서.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print("\n" + "\n".join(lines))
    return df


def collect_and_save_comments(base_path, output_file_prefix, max_workers=5):
    import csv as _csv
    import glob as _glob
    # ★ 기사 본문(text)·이미지는 댓글마다 반복하지 않는다(파일이 수 GB로 커짐). 본문은 articles.csv 와 url 로 연결
    COLS = ["url", "media", "title"] + COMMENT_FIELDS + COMMENT_DERIVED_FIELDS
    total = 0
    comments_dir = os.path.join(base_path, "comments")
    os.makedirs(comments_dir, exist_ok=True)

    # ── 이어하기: 언론사 1곳의 댓글 수집이 전부 끝나면 .done_{언론사} 마커를 남긴다.
    #    재실행 시 마커가 있는 언론사는 통째로 건너뛴다(댓글 재수집 없이).
    #    마커 파일에는 그 언론사에서 수집된 댓글 수를 적어 두고, 재실행 시작
    #    시점에 전부 합산해서 summary/total 수치가 이어하기 전후로 어긋나지
    #    않도록 한다.
    def _done_marker(media_name):
        return os.path.join(comments_dir, f".done_{output_file_prefix}_{media_name}")

    for marker_path in _glob.glob(os.path.join(comments_dir, f".done_{output_file_prefix}_*")):
        try:
            with open(marker_path, encoding="utf-8") as f:
                total += int((f.read() or "0").strip())
        except Exception:
            pass

    for media_name in os.listdir(base_path):
        media_dir = os.path.join(base_path, media_name)
        if not os.path.isdir(media_dir) or media_name == "comments":
            continue
        article_file = os.path.join(media_dir, "articles.csv")
        if not os.path.isfile(article_file):
            continue

        if os.path.isfile(_done_marker(media_name)):
            print(f"  [{media_name}] 이미 댓글 수집 완료 — 건너뜀")
            continue

        # 이 언론사를 (재)수집하기 전에, 예전에 중단된 시도가 남긴 조각 CSV가
        # 있으면 지운다. 안 지우면 아래에서 새로 저장하는 결과와 합쳐져
        # analyze_comment_actions() 집계 시 댓글이 중복 집계된다.
        for stale in _glob.glob(os.path.join(comments_dir, f"{output_file_prefix}_{media_name}*.csv")):
            try:
                os.remove(stale)
            except OSError:
                pass

        print(f"  [{media_name}] 댓글 전수 수집 (workers={max_workers})")
        try:
            df = pd.read_csv(article_file)
        except Exception as e:
            print(f"  로드 오류: {e}"); continue

        rows = [r.to_dict() for _, r in df.iterrows()]
        buf = []
        media_total = 0

        # ── 한 언론사 = 댓글 CSV 1개 ──────────────────────────────────────
        # 메모리 절약을 위해 1만 건마다 디스크로 내보내되, 조각 파일(_partN)을
        # 따로 만들지 않고 하나의 임시파일(.tmp)에 이어 붙인다. 수집이 끝나면
        # os.replace 로 최종 파일명으로 옮기므로, 중간에 끊겨도 최종 경로에는
        # 완성된 파일만 존재한다.
        final_path = os.path.join(comments_dir, f"{output_file_prefix}_{media_name}.csv")
        tmp_path = final_path + ".tmp"
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        wrote_any = False

        def _flush(chunk):
            nonlocal wrote_any
            pd.DataFrame(chunk, columns=COLS).to_csv(
                tmp_path, mode="a" if wrote_any else "w", header=not wrote_any, index=False,
                encoding="utf-8" if wrote_any else "utf-8-sig",   # BOM은 파일 맨 앞에 한 번만
                quoting=_csv.QUOTE_ALL)
            wrote_any = True

        with tqdm(total=len(rows), desc=f"{media_name} 댓글",
                  bar_format="{l_bar}{bar:30}{r_bar}", ncols=80) as pbar:
            with ThreadPoolExecutor(max_workers=max_workers) as ex:
                futs = {ex.submit(_with_kw, getattr(_CTX, "kw", None), _comment_worker, (r, media_name)): r for r in rows}
                for fut in _as_completed(futs):
                    buf.extend(fut.result())
                    pbar.update(1)
                    if len(buf) >= 10000:
                        _flush(buf)
                        total += len(buf); media_total += len(buf); buf = []

        if buf:
            _flush(buf)
            total += len(buf); media_total += len(buf)
        if wrote_any:
            os.replace(tmp_path, final_path)
            _write_excel_copy(final_path)   # 엑셀용 사본 (…_{언론사}.xlsx)

        # 댓글이 0건이었던 언론사도 "완료"로 표시해야 다음 실행에서 매번
        # 다시 (빈) 수집을 반복하지 않는다.
        _atomic_write_text(_done_marker(media_name), str(media_total))
        chk = _write_comment_check(comments_dir, output_file_prefix, media_name,
                                   [r.get("url", "") for r in rows])
        if chk:
            print(f"  ✓ {media_name}: {media_total:,}개 저장 · 네이버 집계 대비 "
                  f"댓글 {chk['got_comment']:,}/{chk['naver_comment']:,}, 답글 {chk['got_reply']:,}/{chk['naver_reply']:,}"
                  + (f" · ⚠️ 덜 받은 기사 {chk['incomplete']}건" if chk['incomplete'] else " · 전수 ✓"))
        else:
            print(f"  ✓ {media_name}: {media_total}개 저장")

    with open(os.path.join(comments_dir, f"{output_file_prefix}_summary.txt"),
              "w", encoding="utf-8") as f:
        f.write(f"총 댓글: {total}\n완료: {time.strftime('%Y-%m-%d %H:%M:%S')}\nworkers: {max_workers}\n")
    print(f"\n★ 총 {total}개 댓글 수집 완료 → {comments_dir}")

    # ── 조치유형 분류 · 생존시간 분석 (방금 저장한 댓글 CSV 전체를 다시 읽어 집계) ──
    analyze_comment_actions(comments_dir, output_file_prefix)


def analyze_comment_actions(comments_dir, output_file_prefix):
    """comments_dir 에 저장된 댓글 CSV 를 모두 모아 조치유형·생존시간을 집계하고
    콘솔에 출력 + {prefix}_action_summary.txt 로 저장한다."""
    import glob
    csv_files = [p for p in glob.glob(os.path.join(comments_dir, f"{output_file_prefix}_*.csv"))
                 if not p.endswith("_summary.txt")]
    if not csv_files:
        return

    dfs = []
    for p in csv_files:
        try:
            dfs.append(pd.read_csv(p))
        except Exception as e:
            print(f"  [분석] 로드 오류: {p}: {e}")
    if not dfs:
        return
    cd = pd.concat(dfs, ignore_index=True)
    if "action_type" not in cd.columns or len(cd) == 0:
        return

    lines = []
    lines.append(f"댓글 총 {len(cd):,}건")

    lines.append("\n[조치유형]")
    for k, v in cd["action_type"].value_counts().items():
        lines.append(f"  {k:20s} {v:6,}")

    lines.append("\n[판정근거]")
    for k, v in cd["action_source"].value_counts().items():
        lines.append(f"  {k:20s} {v:6,}")

    cb = cd[cd["action_type"] == "클린봇"]
    if len(cb):
        n = (cb["cleanbot_original"].astype(str).str.strip() != "").sum()
        lines.append(f"\n[클린봇] {len(cb):,}건 → 가려진 원문 복원 {n:,}건 "
                      f"({n/len(cb)*100:.1f}%)")

    sv = pd.to_numeric(cd["survival_min"], errors="coerce").dropna()
    if len(sv):
        lines.append(f"\n[생존시간] 작성 → 조치까지  n={len(sv):,}")
        lines.append(f"  중앙 {sv.median():,.0f}분 ({sv.median()/1440:.1f}일) · "
                      f"평균 {sv.mean():,.0f}분")
        lines.append(f"  1시간내 {(sv<60).mean()*100:.1f}% · 1일내 {(sv<1440).mean()*100:.1f}% · "
                      f"7일내 {(sv<10080).mean()*100:.1f}%")
        g = (cd.assign(_s=pd.to_numeric(cd["survival_min"], errors="coerce"))
               .dropna(subset=["_s"]).groupby("action_type")["_s"]
               .agg(["count", "median", "mean"]).round(1))
        lines.append("\n[조치유형별 생존시간(분)]")
        lines.append(g.to_string())

    report = "\n".join(lines)
    print("\n" + report)
    with open(os.path.join(comments_dir, f"{output_file_prefix}_action_summary.txt"),
              "w", encoding="utf-8") as f:
        f.write(report + "\n")
    print(f"\n★ 조치유형·생존시간 분석 저장 → "
          f"{os.path.join(comments_dir, output_file_prefix + '_action_summary.txt')}")



# ══════════════════════════════════════════════════════════════════════════════
# 전량 수집 모드 (collect_mode="complete")
# ──────────────────────────────────────────────────────────────────────────────
# 네이버 검색은 일부 기사를 아예 보여주지 않는다(2026.09 실측: 조선일보·이재명·
# 9/22, 본문에 검색어가 있는 34건 중 사설 1건이 검색 결과에 없음). 그래서 검색을
# 거치지 않고 "언론사의 기간 내 모든 기사"를 열어 본문에서 직접 검색어를 찾는다.
#
# 근거(실측):
#   · 네이버 기사번호(aid)는 언론사마다 1씩 증가하며 날짜가 바뀌어도 끊기지 않는다.
#     (연합뉴스 9/14 마지막 16310269 → 9/15 첫 16310270, 하루 2,992건 빈 번호 0)
#   · 그래서 기간 첫날의 첫 번호 ~ 마지막날의 끝 번호만 알면(목록 요청 2번)
#     그 사이 번호를 전부 열면 빠지는 기사가 없다.
#
# 속도:
#   · 1단계(선별): 모든 기사를 동시에 여러 개 열어 제목+본문만 가볍게 추출(기사당 ~1ms)
#     → 16개 동시 ≈ 27건/초, 32개 동시 ≈ 48건/초 (차단 없음 확인)
#   · 결과는 _cache 폴더에 1,000번호 단위 블록으로 저장 → 중단 후 이어하기 가능,
#     키워드를 바꿔 다시 돌리거나 기간이 겹치면 다시 받지 않는다.
#   · 2단계(상세): 검색어가 들어 있는 기사만 기존 collect_article_details 로 상세 수집
# ══════════════════════════════════════════════════════════════════════════════
import gzip as _gzip
import threading as _cthreading
from concurrent.futures import ThreadPoolExecutor as _CPool

COMPLETE_BLOCK = 1000          # 캐시 블록 크기(기사번호 개수)
_C_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
         "(KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36")
_C_TL = _cthreading.local()
_C_DIC = re.compile(r'<article[^>]*id="dic_area".*?</article>', re.S)
_C_TIT = re.compile(r'<h2[^>]*id="title_area"[^>]*>(.*?)</h2>', re.S)
_C_DT = re.compile(r'_ARTICLE_DATE_TIME[^>]*data-date-time="([^"]+)"')


def _c_session():
    if not hasattr(_C_TL, "s"):
        s = requests.Session()
        s.headers.update({"User-Agent": _C_UA, "Accept-Language": "ko-KR,ko;q=0.9"})
        ad = requests.adapters.HTTPAdapter(pool_connections=4, pool_maxsize=4)
        s.mount("https://", ad)
        _C_TL.s = s
    return _C_TL.s


def _c_list_edge(oid, day, newest=True):
    """언론사 oid 의 day(YYYYMMDD) 기사 중 가장 최신(newest=True) / 가장 오래된 기사번호."""
    page = 1 if newest else 99999          # 99999 → 네이버가 마지막 페이지로 맞춰 준다
    r = _c_session().get("https://news.naver.com/main/list.naver",
                         params=dict(mode="LPOD", mid="sec", oid=oid, date=day, page=page), timeout=15)
    # ★ 페이지 옆 '최신 기사' 영역에도 오늘자 기사 링크가 있으므로, 날짜별 목록 영역에서만 번호를 읽는다
    soup = BeautifulSoup(r.text, "html.parser")
    ids = [int(m.group(1)) for a in soup.select("ul.type06_headline li dl dt a, ul.type06 li dl dt a")
           for m in [re.search(rf"article/{oid}/(\d+)", a.get("href", ""))] if m]
    if not ids:
        return None
    return max(ids) if newest else min(ids)


def _c_aid_range(oid, start_date, end_date):
    """기간의 (첫 기사번호, 마지막 기사번호). 기사가 없는 날(주말 등)은 건너뛴다."""
    from datetime import datetime, timedelta
    d0 = datetime.strptime(start_date, "%Y%m%d")
    d1 = datetime.strptime(end_date, "%Y%m%d")
    lo = hi = None
    d = d0
    while d <= d1 and lo is None:
        lo = _c_list_edge(oid, d.strftime("%Y%m%d"), newest=False); d += timedelta(days=1)
    d = d1
    while d >= d0 and hi is None:
        hi = _c_list_edge(oid, d.strftime("%Y%m%d"), newest=True); d -= timedelta(days=1)
    # 마지막 날 밤늦게 입력됐지만 자정 넘어 네이버에 등록된 기사는 다음 날 목록 맨 앞에 붙는다.
    # → 다음 날 첫 번호부터 200번까지 더 열어 보고, 기사 자체의 입력일로 기간을 거른다.
    if hi is not None:
        try:
            nlo = _c_list_edge(oid, (d1 + timedelta(days=1)).strftime("%Y%m%d"), newest=False)
            nhi = _c_list_edge(oid, (d1 + timedelta(days=1)).strftime("%Y%m%d"), newest=True)
            if nlo and nhi and nlo > hi - 1:
                hi = max(hi, min(nhi, nlo + 200))
        except Exception:
            pass
    return lo, hi


def _c_html_text(frag):
    return re.sub(r"\s+", " ", BeautifulSoup(frag or "", "html.parser").get_text(" ", strip=True)).strip()


# ── 전량 모드 차단 대응 (2026.09.30) ─────────────────────────────────────────────
# 이전에는 기사 화면이 403/429(일시 차단)를 돌려주면 몇 번 재시도한 뒤 "없는 번호"로 캐시에
# 영구 저장했다 → 차단 시간 동안의 기사가 조용히 빠질 수 있었다(전수성 훼손).
# 이제는
#   · 404(네이버가 '없는 기사'라고 명시)만 없는 번호로 기록한다.
#   · 403/429 는 모든 스레드가 함께 쉬었다가(1→3→5→10→15→30분) 다시 받고, 차단 뒤에는
#     요청 속도를 스스로 낮춘다.
#   · 끝내 받지 못한 번호가 하나라도 있는 블록은 저장하지 않는다 → 다시 Run All 하면 그 블록만 다시 받는다.
_C_LOCK = _cthreading.Lock()
_C_STATE = {"cool_until": 0.0, "level": 0, "pace": 0.0, "blocks": 0}


def _c_wait():
    while True:
        d = _C_STATE["cool_until"] - time.time()
        if d <= 0:
            break
        time.sleep(min(d, 5))
    if _C_STATE["pace"]:
        time.sleep(_C_STATE["pace"])


def _c_blocked(status):
    """차단 감지: 모든 스레드를 함께 쉬게 하고, 이후 요청 속도를 낮춘다."""
    with _C_LOCK:
        now = time.time()
        if now < _C_STATE["cool_until"]:           # 이미 다른 스레드가 알렸음
            return
        lv = min(_C_STATE["level"], len(BLOCK_WAITS) - 1)
        wait = BLOCK_WAITS[lv]
        _C_STATE.update(cool_until=now + wait, level=_C_STATE["level"] + 1,
                        pace=min(2.0, (_C_STATE["pace"] or 0.1) * 2), blocks=_C_STATE["blocks"] + 1)
        print(f"\n  ⚠️ 네이버 기사 화면 응답 {status} (일시 차단) — 모두 {wait // 60}분 쉬고 속도를 낮춰 재개 "
              f"(스레드당 {_C_STATE['pace']:.1f}초 간격)")


def _c_ok():
    """차단 없이 10분이 지나면 대기 단계와 속도 제한을 한 단계씩 되돌린다."""
    now = time.time()
    if (_C_STATE["level"] or _C_STATE["pace"]) and now > max(_C_STATE["cool_until"], _C_STATE.get("relaxed", 0)) + 600:
        with _C_LOCK:
            _C_STATE["level"] = max(0, _C_STATE["level"] - 1)
            _C_STATE["pace"] = 0.0 if _C_STATE["pace"] <= 0.2 else _C_STATE["pace"] / 2
            _C_STATE["relaxed"] = now


def _c_screen_one(oid, aid, tries=4):
    """기사 1건을 가볍게 열어 {aid,url,kind,date,title,text} 반환.
    없는 번호(404 등)면 missing=True, 차단·네트워크 오류로 끝내 못 받으면 error=True."""
    aid10 = f"{aid:010d}"
    url = f"https://n.news.naver.com/mnews/article/{oid}/{aid10}"
    last, blocked_n = None, 0
    k = 0
    while k < tries:
        _c_wait()
        try:
            r = _c_session().get(url, timeout=20)
            last = r.status_code
            if r.status_code in (403, 429, 503):
                _c_blocked(r.status_code); blocked_n += 1
                if blocked_n > len(BLOCK_WAITS) + 2:
                    break
                continue                              # 차단은 재시도 횟수로 세지 않는다
            if r.status_code == 404:
                return {"aid": aid10, "url": url, "missing": True, "status": "404"}
            host = r.url.split("/")[2] if r.status_code == 200 else ""
            if "sports.naver.com" in host or "entertain.naver.com" in host:
                svc = "sports" if "sports" in host else "entertain"
                rr = _c_session().get(f"https://api-gw.{svc}.naver.com/news/article/{oid}/{aid10}",
                                      headers={"Referer": f"https://m.{svc}.naver.com/"}, timeout=20)
                if rr.status_code in (403, 429, 503):
                    _c_blocked(rr.status_code); blocked_n += 1; continue
                a = rr.json()["result"]["articleInfo"]["article"]
                dt = str(a.get("serviceDatetime", ""))
                _c_ok()
                return {"aid": aid10, "url": url, "kind": svc, "date": dt[:10].replace("-", ""),
                        "datetime": dt, "title": a.get("title", ""), "text": _c_html_text(a.get("content", ""))}
            if r.status_code == 200:
                m = _C_DIC.search(r.text); t = _C_TIT.search(r.text); d = _C_DT.search(r.text)
                if m or t:
                    dt = d.group(1) if d else ""
                    _c_ok()
                    return {"aid": aid10, "url": url, "kind": "news", "date": dt[:10].replace("-", ""),
                            "datetime": dt, "title": _c_html_text(t.group(1)) if t else "",
                            "text": _c_html_text(m.group(0)) if m else ""}
        except Exception as e:
            last = type(e).__name__
        k += 1
        time.sleep(1.5 * k)
    if last == 200:        # 정상 응답인데 기사 형태가 아님(삭제·비공개 안내 화면 등)
        return {"aid": aid10, "url": url, "missing": True, "status": "200-noarticle"}
    if isinstance(last, int) and last >= 500:
        # 기사 화면이 계속 500을 내는 번호가 있다(2026.09 실측: 경향 0003370089).
        # 네이버 기사 API로 한 번 더 확인해 '존재하지 않는 기사'면 없는 번호, 기사가 있으면 API 내용으로 채운다.
        try:
            rr = _c_session().get(f"https://api-gw.sports.naver.com/news/article/{oid}/{aid10}",
                                  headers={"Referer": "https://m.sports.naver.com/"}, timeout=20)
            if rr.status_code == 404:
                return {"aid": aid10, "url": url, "missing": True, "status": f"{last}-api404"}
            if rr.status_code == 200:
                a = ((rr.json().get("result") or {}).get("articleInfo") or {}).get("article") or {}
                if a:
                    dt = str(a.get("serviceDatetime", ""))
                    return {"aid": aid10, "url": url, "kind": "news", "date": dt[:10].replace("-", ""),
                            "datetime": dt, "title": a.get("title", ""), "text": _c_html_text(a.get("content", ""))}
        except Exception:
            pass
    return {"aid": aid10, "url": url, "error": True, "status": str(last)}


def _c_block_path(cache_dir, oid, b):
    # v2: 차단을 '없는 번호'로 저장하지 않는 버전. 이전 버전 블록은 다시 받는다(불러올 때는 함께 쓴다).
    return os.path.join(cache_dir, oid, f"{b:010d}.v2.jsonl.gz")


def build_complete_cache(media_name, news_office_id, start_date, end_date, cache_dir, workers=32):
    """언론사 1곳의 기간 내 모든 기사를 선별용으로 받아 cache_dir 에 저장(이미 있는 블록은 건너뜀)."""
    oid = f"{news_office_id - 1000:03d}"
    os.makedirs(os.path.join(cache_dir, oid), exist_ok=True)
    lo, hi = _c_aid_range(oid, start_date, end_date)
    if lo is None or hi is None or hi < lo:
        print(f"  ⚠️ [{media_name}] 기간 내 기사 목록을 찾지 못함 (oid={oid})")
        return True
    blocks = list(range(lo // COMPLETE_BLOCK, hi // COMPLETE_BLOCK + 1))
    todo = [b for b in blocks if not os.path.isfile(_c_block_path(cache_dir, oid, b))]
    est_h = len(todo) * COMPLETE_BLOCK / 40 / 3600
    print(f"  [{media_name}] 기사번호 {lo}~{hi} (약 {hi - lo + 1:,}건) · 블록 {len(blocks)}개 중 "
          f"{len(blocks) - len(todo)}개는 이미 받음" + (f" · 예상 약 {est_h:.1f}시간" if todo else ""))
    if not todo:
        return True
    total_missing, unsaved = 0, 0
    with tqdm(total=len(todo) * COMPLETE_BLOCK, desc=f"{media_name} 전체기사 선별", unit="건") as pbar, \
            _CPool(max_workers=workers) as ex:
        for b in todo:
            aids = range(b * COMPLETE_BLOCK, (b + 1) * COMPLETE_BLOCK)
            recs = []
            for rec in ex.map(lambda a: _c_screen_one(oid, a), aids):
                recs.append(rec); pbar.update(1)
            # 오류·애매한 번호는 잠시 뒤 두 번 더 확인
            for _round in range(2):
                redo = [i for i, r in enumerate(recs)
                        if r.get("error") or (r.get("missing") and not str(r.get("status", "")).endswith("404"))]
                if not redo:
                    break
                time.sleep(5)
                again = list(ex.map(lambda i: _c_screen_one(oid, int(recs[i]["aid"])), redo))
                for i, r in zip(redo, again):
                    recs[i] = r
            # 블록 끝이 아직 발행 전 번호(미래)라면 저장하지 않아야 나중에 채워진다
            cut = (hi % COMPLETE_BLOCK) + 1
            if b == hi // COMPLETE_BLOCK and all(r.get("missing") or r.get("error") for r in recs[cut:]):
                recs_to_save, partial = recs[:cut], True
            else:
                recs_to_save, partial = recs, False
            if any(r.get("error") for r in recs_to_save):
                unsaved += 1
                pbar.set_postfix({"없는번호": total_missing, "미저장블록": unsaved})
                continue                              # 다음 Run All 때 이 블록만 다시 받는다
            total_missing += sum(1 for r in recs_to_save if r.get("missing"))
            path = _c_block_path(cache_dir, oid, b)
            if partial:
                path = path.replace(".jsonl.gz", f"_upto{hi}.partial.jsonl.gz")
            tmp = path + ".tmp"
            with _gzip.open(tmp, "wt", encoding="utf-8") as f:
                for r in recs_to_save:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
            os.replace(tmp, path)
            pbar.set_postfix({"없는번호": total_missing, "미저장블록": unsaved})
    if unsaved:
        print(f"  ❌ [{media_name}] 차단·서버 오류가 계속되어 {unsaved}개 블록({unsaved * COMPLETE_BLOCK:,}번호)을 받지 못했습니다. "
              f"같은 설정으로 다시 Run All 하면 그 블록만 이어서 받습니다.")
        return False
    print(f"  ✓ [{media_name}] 선별 완료 · 없는 번호 {total_missing}건(삭제·비공개 등)")
    return True


def load_complete_cache(news_office_id, start_date, end_date, cache_dir):
    """캐시에서 기간(start~end) 안의 기사 레코드만 모아 반환."""
    import glob as _g
    oid = f"{news_office_id - 1000:03d}"
    out, missing = {}, 0
    for p in sorted(_g.glob(os.path.join(cache_dir, oid, "*.jsonl.gz"))):
        with _gzip.open(p, "rt", encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                if r.get("missing"):
                    missing += 1; continue
                if start_date <= r.get("date", "") <= end_date:
                    out[r["aid"]] = r
    return list(out.values())


def complete_matches(records, query):
    """제목·본문(부제·사진설명 포함)에 검색어가 글자 그대로 들어 있는 기사."""
    kw = query.strip('"').strip()
    return [r for r in records if kw in r.get("title", "") or kw in r.get("text", "")]


_BL_ROLE = r"(?:선임기자|수석기자|전문기자|객원기자|인턴기자|기자|특파원|논설위원|논설실장|주필|편집국장|통신원|기상캐스터|리포터)"
_BL_NAME = r"[가-힣]{2,4}"
_BL_NAMES = rf"{_BL_NAME}(?:\s*[·ㆍ,]\s*{_BL_NAME})*"
_BL_EMAIL = re.compile(r"[\w.\-+]+@[\w\-]+(?:\.[\w\-]+)+")
_BL_STOP = {"취재", "사진", "영상", "담당", "출입", "현장", "동료", "해당", "일부", "외신", "방송", "신문", "여러", "다른", "모든", "전직", "현직"}


def _byline_from_text(fulltext):
    """본문에서 바이라인(기자명·이메일) 추출. 2026.09 현재 네이버 기사 화면에 별도 바이라인
    영역이 없어, 본문 첫머리('(서울=연합뉴스) 홍길동 기자 =' / '[매체 홍길동 기자]')나
    맨 끝('홍길동 기자 hong@…')에 적힌 것을 위치 조건을 엄격히 걸어 읽는다."""
    t = (fulltext or "").strip()
    head, tail = t[:160], t[-260:]
    found = []
    m = re.search(rf"({_BL_NAMES})\s*({_BL_ROLE})\s*[=\]\)]", head)
    if m:
        found.append((m.group(1), m.group(2)))
    for m in re.finditer(rf"({_BL_NAMES})\s*({_BL_ROLE})(?=\s*(?:[\w.\-+]+@|[\(\[/=,·]|$|\n))", tail):
        found.append((m.group(1), m.group(2)))
    names = []
    for grp, role in found:
        for n in re.split(r"\s*[·ㆍ,]\s*", grp):
            if n and n not in _BL_STOP and f"{n} {role}" not in names:
                names.append(f"{n} {role}")
    emails = list(dict.fromkeys(_BL_EMAIL.findall(tail)))
    return ", ".join(names[:5]), ", ".join(emails[:3])


# ── 네이버 기사 API: 기자 정보 + 스포츠·연예 기사 보완 ─────────────────────────────
# ★ 2026.09 실측: 네이버 기사 화면에서 바이라인 영역(span.byline_s, _JOURNALIST_CARD)이 사라져
#   writer·journalist_id 가 모든 기사에서 비어 있었다. 대신 네이버 기사 API
#   (api-gw.sports.naver.com/news/article/{oid}/{aid})가 스포츠 기사뿐 아니라 일반 기사에도
#   기자 카드(이름·이메일·기자ID)와 본문을 돌려준다.
#   또 스포츠·연예 기사는 일반 기사 화면이 아니라 파싱이 비므로 같은 API로 제목·본문·날짜를 채운다.
_ART_API_HOSTS = ("sports", "entertain")
_BL_ROLE_RE = re.compile(r"(기자|특파원|논설위원|논설실장|주필|편집국장|통신원|에디터|위원|캐스터|리포터)")
_BL_ORG_RE = re.compile(r"(제공|일보|신문|뉴스|방송|통신|리포터|편집부|취재팀|팀$|DB$)")


def _naver_article_api(oid, aid, tries=3):
    """네이버 기사 API의 articleInfo(dict). 실패하면 {}."""
    for k in range(tries):
        for svc in _ART_API_HOSTS:
            try:
                r = requests.get(f"https://api-gw.{svc}.naver.com/news/article/{oid}/{aid}",
                                 headers={"User-Agent": _C_UA, "Referer": f"https://m.{svc}.naver.com/",
                                          "Accept-Language": "ko-KR,ko;q=0.9"}, timeout=15)
                if r.status_code == 200:
                    ai = (r.json().get("result") or {}).get("articleInfo") or {}
                    if ai.get("article"):
                        return ai
                elif r.status_code in (403, 429):
                    time.sleep(10 * (k + 1))
            except Exception:
                pass
        time.sleep(1.5 * (k + 1))
    return {}


def _byline_info(ai):
    """articleInfo → (기자명들, 이메일들, 기자ID들, 네이버 등록기자 여부)"""
    jc = ai.get("journalistCards") or {}
    cards = jc.get("cards") or []
    jcards = [c for c in cards if c.get("type") == "JOURNALIST_CARD"]
    names = [c.get("name", "").strip() for c in cards if c.get("type") == "UNIFORM_CARD" and c.get("name")]
    if not names:
        names = [c.get("name", "").strip() for c in jcards if c.get("name")]
    bylines = jc.get("bylines") or []
    if not names and bylines:
        names = [re.sub(r"[\(\[]?[\w.\-+]+@[\w.\-]+[\)\]]?", "", b).strip(" ()[]/") for b in bylines]
    emails = [c.get("email", "") for c in cards if c.get("email")]
    for b in bylines:
        emails += _BL_EMAIL.findall(b)
    emails = [e.lower() for e in _BL_EMAIL.findall(" ".join(emails))]   # 보이지 않는 문자까지 걸러 정규화
    ids = [str(c.get("journalistId")) for c in jcards if c.get("journalistId")]
    # '광주광역시=조홍복 기자' 처럼 앞에 붙은 지역·매체 표기는 떼어 같은 기자가 같은 이름으로 모이게 한다
    names = [re.sub(r"^[^=\s]{1,15}\s*=\s*", "", n).strip() for n in names]
    return (", ".join(dict.fromkeys(n for n in names if n)),
            ", ".join(dict.fromkeys(e for e in emails if e)),
            "|".join(dict.fromkeys(ids)), bool(jcards))


# ── 공감 반응 · 댓글 집계/작성자 통계 (2026.09 실측으로 경로 확인) ────────────────────
# ★ 네이버 기사 화면의 공감 반응과 댓글 작성 통계는 페이지 원본에 없고 따로 불러온다.
#   · 공감 반응: route-like.naver.com/v1/search/contents  (화면 값과 일치 확인)
#   · 댓글 집계·허용 상태·작성자 성별/연령 비율: 댓글 목록 API(initialize=true)의 count / exposureConfig / graph
# ★ 반응 버튼은 시기에 따라 다르다.
#   ~2022년경: 좋아요(like)·훈훈해요(warm)·슬퍼요(sad)·화나요(angry)·후속기사 원해요(want)
#   현재   : 쏠쏠정보(useful)·흥미진진(wow)·공감백배(touched)·분석탁월(analytical)·후속강추(recommend)
#   전환기 기사는 두 가지가 섞여 있으므로 10가지를 모두 따로 저장한다.
_REACTION_COLS = {
    "like": "react_like", "warm": "react_warm", "sad": "react_sad", "angry": "react_angry", "want": "react_want",
    "useful": "react_useful", "wow": "react_wow", "touched": "react_touched",
    "analytical": "react_analytic", "recommend": "react_recommend",
}
_AGE_BANDS = ("10", "20", "30", "40", "50", "60", "70")


def _naver_reactions(oid, aid, referer, tries=3):
    """{반응유형: 수}. 반응이 없으면 {}, 끝내 못 받으면 None."""
    for k in range(tries):
        try:
            t = requests.get("https://route-like.naver.com/v1/search/contents",
                             params={"suppress_response_codes": "true", "q": f"NEWS[ne_{oid}_{aid}]",
                                     "isDuplication": "false"},
                             headers={"User-Agent": _C_UA, "Referer": referer}, timeout=15).text
            m = re.search(r"\{.*\}", t, re.S)
            if m:
                cs = json.loads(m.group(0)).get("contents") or []
                if not cs:
                    return {}
                return {r.get("reactionType"): int(r.get("count") or 0) for r in (cs[0].get("reactions") or [])}
        except Exception:
            pass
        time.sleep(2 * (k + 1))
    return None


def _naver_comment_meta(oid, aid, referer, tries=3):
    """댓글 목록 API 첫 호출(initialize=true)의 result: count·exposureConfig·graph. 실패하면 None."""
    params = {"ticket": "news", "templateId": "default_society", "pool": "cbox5", "lang": "ko", "country": "KR",
              "objectId": f"news{oid},{aid}", "pageSize": 1, "indexSize": 10, "page": 1,
              "sort": "FAVORITE", "initialize": "true", "pageType": "more"}
    for k in range(tries):
        try:
            r = requests.get("https://apis.naver.com/commentBox/cbox/web_naver_list_json.json", params=params,
                             headers={"User-Agent": _C_UA, "Referer": referer}, timeout=15)
            if r.status_code == 200:
                j = r.json()
                if j.get("success"):
                    return j.get("result") or {}
            elif r.status_code in (403, 429):
                time.sleep(15 * (k + 1))
        except Exception:
            pass
        time.sleep(2 * (k + 1))
    return None


def _fill_reactions_and_comment_meta(d, oid, aid, referer):
    rx = _naver_reactions(oid, aid, referer)
    if rx is None:                                   # 받지 못함 → 0 이 아니라 빈칸으로 둔다
        for col in _REACTION_COLS.values():
            d[col] = ""
        d["react_total"] = ""; d["react_other"] = ""
    else:
        for t, col in _REACTION_COLS.items():
            d[col] = rx.get(t, 0)
        d["react_total"] = sum(rx.values())
        other = {k: v for k, v in rx.items() if k not in _REACTION_COLS}
        d["react_other"] = json.dumps(other, ensure_ascii=False) if other else ""

    cm = _naver_comment_meta(oid, aid, referer)
    keys = ["comment_n", "reply_n", "comment_exposed_n", "comment_del_by_writer", "comment_del_by_rule",
            "comment_status", "comment_allow", "comment_stats_available",
            "commenter_male_pct", "commenter_female_pct"] + [f"commenter_age{a}_pct" for a in _AGE_BANDS]
    if cm is None:
        for k in keys:
            d[k] = ""
        return
    cnt, ex, g = cm.get("count") or {}, cm.get("exposureConfig") or {}, cm.get("graph") or {}
    status = ex.get("status") or ""
    d.update({
        "comment_n": int(cnt.get("comment") or 0), "reply_n": int(cnt.get("reply") or 0),
        "comment_exposed_n": int(cnt.get("exposeCount") or 0),
        "comment_del_by_writer": int(cnt.get("delCommentByUser") or 0),
        "comment_del_by_rule": int(cnt.get("delCommentByMon") or 0),
        "comment_status": status, "comment_allow": 1 if status == "COMMENT_ON" else 0,
    })
    gen = g.get("gender") or {}
    old = {str(o.get("age")): o.get("value") for o in (g.get("old") or [])}
    # 네이버는 댓글이 어느 정도 쌓인 기사에만 작성자 통계를 보여준다 → 없으면 빈칸
    d["comment_stats_available"] = 1 if gen else 0
    d["commenter_male_pct"] = gen.get("male", "") if gen else ""
    d["commenter_female_pct"] = gen.get("female", "") if gen else ""
    for a in _AGE_BANDS:
        d[f"commenter_age{a}_pct"] = old.get(a, "") if gen else ""


def collect_article_details(media_name, url, collect_image=True, image_quality='medium'):
    """기사 화면 파싱(_collect_article_details_html) + 네이버 기사 API 보완."""
    d = _collect_article_details_html(media_name, url, collect_image, image_quality)
    d.setdefault("writer_email", "")
    m = re.search(r"article/(\d{3})/(\d{10})", url or "")
    if not m:
        return d
    oid, aid = m.group(1), m.group(2)
    ai = _naver_article_api(oid, aid)
    a = ai.get("article") or {}

    # 스포츠·연예 등 일반 기사 화면이 아닌 기사: 비어 있는 칸만 API 값으로 채운다
    is_se = False
    if a and (not d.get("fulltext") or not d.get("datetime")):
        is_se = not d.get("fulltext")
        _cs = BeautifulSoup(a.get("content", "") or "", "html.parser")
        _boxes = _cs.select(".navernews_end_summary")          # 요약 상자는 본문과 따로
        if _boxes and not d.get("summary_box"):
            d["summary_box"] = " || ".join(_clean_text(e.get_text(" ", strip=True)) for e in _boxes)
            d["has_summary_box"] = 1
        for e in _boxes:
            e.decompose()
        body = re.sub(r"\s+", " ", _cs.get_text(" ", strip=True)).strip()
        dt, mdt = str(a.get("serviceDatetime", "")), str(a.get("modifyDatetime", ""))
        fill = {
            "title": a.get("title", ""), "naver_oid": oid, "naver_aid": aid, "article_id": aid,
            "date": dt[:10], "time": dt[11:16],
            "datetime": f"{dt[:10]}T{dt[11:19]}+0900" if len(dt) >= 19 else dt,
            "modify_date": mdt,
            "modify_datetime": f"{mdt[:10]}T{mdt[11:19]}+0900" if len(mdt) >= 19 else mdt,
            # 스포츠·연예 기사의 rawSection 은 내부 코드값이라 사람이 읽을 수 있는 이름으로 둔다
            "section": "스포츠·연예",
        }
        if not d.get("fulltext") and body:
            fill.update({"text": body, "fulltext": body, "text_len": len(body),
                         "fulltext_available": 1 if len(body) >= 50 else 0})
        for k, v in fill.items():
            if v and not d.get(k):
                d[k] = v

    writer, email, jid, staff = _byline_info(ai) if ai else ("", "", "", False)
    if not writer:                      # API에 기자 정보가 없으면 본문 첫머리·끝의 바이라인을 읽는다
        writer, email2 = _byline_from_text(d.get("fulltext", ""))
        email = email or email2
    _fill_reactions_and_comment_meta(d, oid, aid, url)
    if is_se:
        # 스포츠·연예 기사는 네이버 정책으로 댓글란이 없다(연예 2019.10~, 스포츠 2020.8~).
        # 댓글 서버는 'COMMENT_ON' 을 돌려주지만 화면에는 댓글란 자체가 없으므로 0 으로 둔다.
        d["comment_allow"] = 0
        d["comment_status"] = "SPORTS_ENTERTAIN_NO_COMMENT"
    d["writer"] = writer or d.get("writer", "")
    d["writer_email"] = email
    d["journalist_id"] = jid or d.get("journalist_id", "")

    # 기사 유형 재판정(바이라인 규칙이 기자명을 쓰므로). 네이버 등록기자인데 직함이 없으면 '기자'로 본다
    gw = d["writer"]
    if staff and gw and not _BL_ROLE_RE.search(gw):
        gw += " 기자"
    # 바이라인이 언론사·기관명("조선일보", "삼성전자 제공")이거나 리포터면 '외부필자' 규칙에 걸리지 않게 한다
    if gw and (gw.strip() == str(media_name).strip() or _BL_ORG_RE.search(gw)):
        gw = ""
    if d.get("title"):
        genre, genre_rule = detect_genre(d["title"], d.get("section", ""), gw)
        d.update({"genre": genre, "genre_rule": genre_rule, "genre_label": GENRE_NAME.get(genre, "")})
    return d



def collect_and_save_data(news_offices, query, base_path, start_date, end_date, max_pages=1000, collect_comments=True, collect_image=True, image_quality='medium', separate_image_rows=False, strict_filter=True, comment_workers=5, use_query=True,
                          collect_mode="search", cache_dir=None, detail_workers=8, complete_workers=32):
    # 출력 디렉토리 생성
    os.makedirs(base_path, exist_ok=True)
    effective_strict = strict_filter and use_query  # 키워드 미사용 시 자동 해제

    # 1. 뉴스 기사 수집
    for media_name, media_id in news_offices.items():
        # 언론사별 저장 디렉토리 생성
        media_dir = os.path.join(base_path, media_name)
        os.makedirs(media_dir, exist_ok=True)
        article_file = os.path.join(media_dir, "articles.csv")

        # ── 이어하기: articles.csv 가 이미 있으면 이 언론사는 완전히 끝난 것이므로
        #    (to_csv 는 원자적으로만 최종 경로에 쓰이므로 반쪽짜리 파일일 수 없다)
        #    링크 수집부터 다시 하지 않고 건너뛴다.
        if os.path.isfile(article_file):
            print(f"\n▶ {media_name}: 이미 수집됨 — 건너뜀 ({article_file})")
            continue

        print(f"\n▶ {media_name} 뉴스 수집 시작...")

        # 기사 데이터 수집
        articles = []
        rec_by_url = {}
        if use_query and collect_mode == "complete":
            # 전량 수집: 캐시(없으면 지금 생성)에서 검색어가 들어 있는 기사를 고른다
            cache_dir = cache_dir or os.path.join(os.path.dirname(base_path.rstrip("\\/")), "_cache")
            if not build_complete_cache(media_name, media_id, start_date, end_date, cache_dir, complete_workers):
                print(f"❌ {media_name}: 전체 기사 확인이 끝나지 않아 저장하지 않고 넘어갑니다(다시 실행하면 이어서).")
                continue
            recs = load_complete_cache(media_id, start_date, end_date, cache_dir)
            matched = complete_matches(recs, query)
            print(f"  [전량 수집] 기간 내 기사 {len(recs):,}건 중 '{query.strip(chr(34))}' 포함 {len(matched):,}건")
            links = [r["url"] for r in matched]
            rec_by_url = {r["url"]: r for r in matched}
        elif use_query:
            try:
                links = collect_naver_news_links(
                    query, media_id, start_date, end_date, max_pages,
                    progress_path=os.path.join(media_dir, "_links_progress.json"))
            except LinkCollectionIncomplete as e:
                # 받지 못한 구간이 있으면 articles.csv 를 만들지 않는다 → 다시 실행하면 그 구간부터 이어서 받음
                print(f"❌ {media_name}: 차단이 풀리지 않아 {len(e.failed)}개 구간을 받지 못했습니다 "
                      f"{e.failed[:3]}{' …' if len(e.failed) > 3 else ''} — 이 언론사는 저장하지 않고 넘어갑니다. "
                      f"잠시 뒤 같은 설정으로 다시 실행하면 받은 구간은 건너뛰고 이어서 수집합니다.")
                continue
        else:
            # 키워드 없이 언론사 전체 기사: 기간 내 모든 기사번호를 연 결과(캐시)를 그대로 쓴다
            cache_dir = cache_dir or os.path.join(os.path.dirname(base_path.rstrip("\\/")), "_cache")
            if not build_complete_cache(media_name, media_id, start_date, end_date, cache_dir, complete_workers):
                print(f"❌ {media_name}: 전체 기사 확인이 끝나지 않아 저장하지 않고 넘어갑니다(다시 실행하면 이어서).")
                continue
            recs = load_complete_cache(media_id, start_date, end_date, cache_dir)
            links = [r["url"] for r in recs]
            rec_by_url = {r["url"]: r for r in recs}

        if not links:
            print(f"⚠️ {media_name}에서 수집된 링크가 없습니다.")
            continue

        # tqdm 설정 변경 - 간결한 게이지 바 형식으로 표시 (한 줄로 유지)
        def _detail(url):
            d = collect_article_details(media_name, url, collect_image, image_quality)
            r = rec_by_url.get(url)
            if r and not d.get("title"):
                # 스포츠·연예 기사는 일반 기사 화면이 아니라 상세 파싱이 비므로 선별 단계 값으로 채운다
                d.update({"title": r["title"], "text": r["text"], "fulltext": r["text"],
                          "text_len": len(r["text"]), "fulltext_available": 1 if len(r["text"]) >= 50 else 0,
                          "naver_oid": url.split("/")[-2], "naver_aid": r["aid"],
                          "date": r["datetime"][:10], "datetime": r["datetime"], "section": r["kind"]})
            return d

        # 상세 수집을 detail_workers 개씩 동시에 (순서는 그대로 유지)
        with tqdm(total=len(links), desc=f"{media_name} 기사",
                 bar_format='{l_bar}{bar:30}{r_bar}',
                 ncols=80, position=0, leave=True) as pbar, \
                ThreadPoolExecutor(max_workers=max(1, detail_workers)) as _dex:
            for article_details in _dex.map(_detail, links):

                # 이미지 행 분리 옵션
                if separate_image_rows and article_details.get('images'):
                    # 이미지가 여러 개면 각 이미지마다 별도 행 생성
                    image_urls = article_details['images'].split('|')
                    if image_urls and image_urls[0]:  # 빈 문자열 체크
                        for img_url in image_urls:
                            # 각 이미지마다 새로운 행 생성
                            article_copy = article_details.copy()
                            article_copy['images'] = img_url  # 하나의 이미지만
                            articles.append(article_copy)
                    else:
                        # 이미지가 없으면 그냥 추가
                        articles.append(article_details)
                else:
                    # 기본 모드: 이미지를 | 로 구분해서 한 행에
                    articles.append(article_details)

                pbar.update(1)

        # 기사 데이터 저장
        df = pd.DataFrame(articles)

        # 빈 데이터 필터링
        # ★ 사진 기사는 사진설명을 걷어내면 text 가 비므로, 제목만 있으면 유효로 본다
        valid_articles = df[df['title'].fillna("") != ""]

        # ── 엄격하게 수집 (strict_filter=True 시 검색어 포함 기사만 저장) ──────
        # ★ 버그 수정: title/text 만 검사하면 놓친다.
        #   본문 분해 단계에서 부제(strong.media_end_summary 등)와 기사 서두의
        #   굵은 글씨 리드문("가짜 부제")을 text에서 걷어내는데, 검색어가 하필
        #   그 리드문/부제 안에만 있는 기사가 많으면(예: "5·18" 관련 기사는
        #   서두에 굵게 강조된 리드 문장에 검색어가 오는 경우가 흔하다)
        #   text 에서는 검색어가 사라져 title 도 못 맞고 text 도 못 맞아
        #   전부 0건으로 걸러지는 문제가 생긴다.
        #   → subtitle/captions/subheads 까지 함께 검사해 걷어낸 텍스트에
        #     있던 검색어도 놓치지 않는다. (2번째 코드의 apply_strict_filter 가
        #     title+subtitle+fulltext 를 하나로 합쳐 검사하는 것과 같은 이유)
        if effective_strict and collect_mode != "complete":
            keyword = query.strip('"')  # 검색어에서 따옴표 제거 후 필터 키워드로 사용
            before_count = len(valid_articles)
            hay_cols = [c for c in ("title", "subtitle", "summary_box", "captions", "subheads", "text")
                        if c in valid_articles.columns]
            mask = False
            for col in hay_cols:
                mask = mask | valid_articles[col].str.contains(keyword, na=False, regex=False)
            valid_articles = valid_articles[mask]
            print(f"  [엄격하게 수집] '{keyword}' 포함 기사({'+'.join(hay_cols)} 기준): "
                  f"{before_count}개 → {len(valid_articles)}개")

        # 저장 (임시파일 → os.replace 로 원자적 이동: 중간에 중단돼도
        #        article_file 경로엔 "완전히 끝난 결과"만 존재하게 된다)
        import csv as _csv
        _atomic_write_csv(valid_articles, article_file, index=False, encoding="utf-8-sig", quoting=_csv.QUOTE_ALL)
        _write_excel_copy(article_file)   # 엑셀용 사본 (articles.xlsx)
        print(f"✓ {len(valid_articles)}개의 기사를 {article_file}에 저장했습니다.")

    print("\n★ 모든 뉴스 수집이 완료되었습니다! ★")

    # 2. 댓글 수집 (설정이 활성화된 경우) — 전수 수집 로직 사용
    if collect_comments:
        safe_query = sanitize_filename(query)
        print("\n▶ 댓글 전수 수집을 시작합니다...")
        collect_and_save_comments(base_path, safe_query, max_workers=comment_workers)

def sanitize_filename(filename):
    """
    파일 이름에 사용할 수 없는 문자를 제거하거나 대체하는 함수
    """
    # 윈도우에서 파일 이름으로 사용할 수 없는 문자
    invalid_chars = ['\\', '/', ':', '*', '?', '"', '<', '>', '|']
    for ch in invalid_chars:
        filename = filename.replace(ch, '_')
    return filename.strip() or "전체기사"



# ══════════════════════════════════════════════════════════════════════════════
# 다중 키워드 수집 (키워드별 서브폴더)
# ══════════════════════════════════════════════════════════════════════════════

def _run_single_keyword(args):
    (kw, news_offices, base_path, start_date, end_date,
     max_pages, collect_comments, collect_image,
     image_quality, separate_image_rows, strict_filter, comment_workers, use_query,
     collect_mode, cache_dir, detail_workers, complete_workers) = args

    kw_clean = kw.strip('"').strip()
    _CTX.kw = kw_clean          # 이 스레드의 출력에 [키워드] 태그를 붙인다
    kw_dir   = os.path.join(base_path, kw_clean)
    os.makedirs(kw_dir, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  키워드: [{kw_clean}]  폴더: {kw_dir}")
    print(f"{'='*70}")

    try:
        _run_keyword_body(kw, news_offices, kw_dir, start_date, end_date, max_pages, collect_comments,
                          collect_image, image_quality, separate_image_rows, strict_filter,
                          comment_workers, use_query, collect_mode, cache_dir, detail_workers, complete_workers)
    finally:
        _CTX.kw = None          # 다음 출력(보고서 등)에 키워드 태그가 남지 않도록


def _run_keyword_body(kw, news_offices, kw_dir, start_date, end_date, max_pages, collect_comments,
                      collect_image, image_quality, separate_image_rows, strict_filter,
                      comment_workers, use_query, collect_mode, cache_dir, detail_workers, complete_workers):
    collect_and_save_data(
        news_offices, kw, kw_dir,
        start_date, end_date, max_pages,
        collect_comments, collect_image,
        image_quality, separate_image_rows,
        strict_filter, comment_workers, use_query,
        collect_mode=collect_mode, cache_dir=cache_dir,
        detail_workers=detail_workers, complete_workers=complete_workers,
    )


def collect_multi_keywords(
    queries,
    news_offices,
    base_path,
    start_date,
    end_date,
    max_pages           = 10000,
    collect_comments    = True,
    collect_image        = False,
    image_quality        = "low",
    separate_image_rows  = False,
    strict_filter        = True,
    keyword_workers      = 1,
    comment_workers       = 5,
    use_query            = True,
    collect_mode         = "search",
    complete_workers     = 32,
    detail_workers       = 8,
):
    os.makedirs(base_path, exist_ok=True)
    if not use_query:
        queries = ["전체기사"]   # 키워드 미사용: 언론사 전체 기사(폴더명 placeholder)
    print(f"수집 키워드: {queries}")
    print(f"처리 방식: {'병렬 ' + str(keyword_workers) + '개' if keyword_workers > 1 else '순차'}")
    print(f"수집 방식: {'전량 수집(모든 기사 확인)' if collect_mode == 'complete' and use_query else '네이버 검색'}")

    # 전량 수집: 키워드와 무관하게 언론사별 전체 기사를 먼저 한 번만 받아 둔다(키워드끼리 공유)
    cache_dir = os.path.join(base_path, "_cache")
    if collect_mode == "complete" or not use_query:
        print("\n▶ [전량 수집 1단계] 언론사별 전체 기사 선별 (이미 받은 구간은 건너뜀)")
        _undone = []
        for _mn, _mid in news_offices.items():
            try:
                if not build_complete_cache(_mn, _mid, start_date, end_date, cache_dir, complete_workers):
                    _undone.append(_mn)
            except Exception as _e:
                print(f"  ❌ [{_mn}] 선별 중 오류: {_e} — 다시 실행하면 이어서 받습니다."); _undone.append(_mn)
        if _undone:
            print(f"  ⚠️ 선별이 끝나지 않은 언론사: {_undone} (2단계에서 건너뜀, 다시 Run All 하면 이어서)")
        print("▶ [전량 수집 2단계] 키워드별로 해당 기사 상세·댓글 수집")

    args_list = [
        (kw, news_offices, base_path, start_date, end_date,
         max_pages, collect_comments, collect_image,
         image_quality, separate_image_rows, strict_filter, comment_workers, use_query,
         collect_mode, cache_dir, detail_workers, complete_workers)
        for kw in queries
    ]

    if keyword_workers == 1:
        for args in args_list:
            _run_single_keyword(args)
    else:
        workers = min(keyword_workers, len(queries))
        with ThreadPoolExecutor(max_workers=workers) as ex:
            futs = [ex.submit(_run_single_keyword, a) for a in args_list]
            for fut in _as_completed(futs):
                try:
                    fut.result()
                except Exception as e:
                    print(f"키워드 오류: {e}")

    print(f"\n★ 전체 키워드 수집 완료 ★")
    write_run_report(base_path, queries, news_offices, start_date, end_date, use_query)


# ══════════════════════════════════════════════════════════════════════════════
# 언론사 목록(네이버 언론사 코드 = 1000 + oid). 2026.09 네이버 언론사 이름과 전부 대조 확인함.
# ══════════════════════════════════════════════════════════════════════════════
NEWS_OFFICES = {
    "KBS": 1056,
    "MBC": 1214,
    "SBS": 1055,
    "YTN": 1052,
    "연합뉴스TV": 1422,
    "MBN": 1019,
    "TV조선": 1448,
    "채널A": 1449,
    "JTBC": 1437,
    "SBS Biz": 1374,
    "한국경제tv": 1004,
    "경향신문": 1032,
    "국민일보": 1005,
    "동아일보": 1020,
    "문화일보": 1021,
    "서울신문": 1081,
    "세계일보": 1022,
    "조선일보": 1023,
    "중앙일보": 1025,
    "한겨레": 1028,
    "한국일보": 1469,
    "매일경제": 1009,
    "머니투데이": 1008,
    "서울경제": 1011,
    "아시아경제": 1277,
    "이데일리": 1018,
    "파이낸셜뉴스": 1014,
    "한국경제": 1015,
    "헤럴드경제": 1016,
    "조선비즈": 1366,
    "한경비즈니스": 1050,
    "비즈워치": 1648,
    "매경이코노미": 1024,
    "이코노미스트": 1243,
    "코리아헤럴드": 1044,
    "연합뉴스": 1001,
    "뉴시스": 1003,
    "뉴스1": 1421,
    "오마이뉴스": 1047,
    "프레시안": 1002,
    "미디어오늘": 1006,
    "노컷뉴스": 1079,
    "데일리안": 1119,
    "아이뉴스24": 1031,
    "뉴스타파": 1607,
    "더팩트": 1629,
    "한겨레21": 1036,
    "주간경향": 1033,
    "주간동아": 1037,
    "주간조선": 1053,
    "시사저널": 1586,
    "시사IN": 1308,
    "신동아": 1262,
    "부산일보": 1082,
    "국제신문": 1658,
    "경기일보": 1666,
    "강원도민일보": 1654,
    "강원일보": 1087,
    "대전일보": 1656,
    "매일신문": 1088,
    "제주일보": 1084,
    "kbc광주방송": 1660,
    "CJB청주방송": 1655,
    "전주MBC": 1659,
    "대구MBC": 1657,
    "지디넷코리아": 1092,
    "전자신문": 1030,
    "디지털타임스": 1029,
    "블로터": 1293,
    "디지털데일리": 1138,
    "농민신문": 1662,
    "코메디닷컴": 1296,
    "헬스조선": 1346,
    "스포츠조선": 1076,
    "스포츠동아": 1382,
    "스포츠서울": 1468,
    "일간스포츠": 1241,
    "OSEN": 1109,
    "스타뉴스": 1108,
}


def verify_offices(news_offices):
    """선택한 언론사 코드가 실제 그 언론사인지 네이버에서 확인(언론사당 요청 1번)."""
    norm = lambda x: re.sub(r"[\s·]", "", str(x)).lower()
    bad = []
    for name, code in news_offices.items():
        oid = f"{int(code) - 1000:03d}"
        try:
            t = requests.get("https://news.naver.com/main/list.naver",
                             params=dict(mode="LPOD", mid="sec", oid=oid),
                             headers={"User-Agent": _C_UA}, timeout=15).text
            m = re.search(r"<title>\s*(.*?)\s*홈\s*:", t)
            real = m.group(1).strip() if m else ""
            if real and norm(real) != norm(name):
                bad.append(f"{name}(코드 {code}) → 네이버에서는 '{real}'")
        except Exception:
            pass
        time.sleep(0.3)
    if bad:
        print("⚠️ 언론사 코드 확인 필요:\n   " + "\n   ".join(bad))
    else:
        print(f"✓ 언론사 {len(news_offices)}곳 코드 확인 완료")


def run_collection(검색어, 언론사, 시작일, 종료일, 저장폴더, 수집방식="검색", 댓글수집=True,
                   요청간격_초=2.0, 엄격필터=True, 이미지수집=False, 동시댓글작업=5):
    """설정 칸의 값으로 수집 전체를 실행한다."""
    global SEARCH_DELAY, SEARCH_WINDOW_DAYS
    d0 = re.sub(r"\D", "", str(시작일)); d1 = re.sub(r"\D", "", str(종료일))
    if len(d0) != 8 or len(d1) != 8 or d0 > d1:
        raise ValueError(f"날짜를 확인하세요: 시작일={시작일}, 종료일={종료일} (예: '2025-05-01')")
    if isinstance(언론사, str):
        언론사 = list(NEWS_OFFICES) if 언론사.strip() == "전체" else [언론사]
    unknown = [n for n in 언론사 if n not in NEWS_OFFICES]
    if unknown:
        raise ValueError(f"목록에 없는 언론사: {unknown}\n가능한 이름: {', '.join(NEWS_OFFICES)}")
    offices = {n: NEWS_OFFICES[n] for n in 언론사}
    mode = {"검색": "search", "전량": "complete", "search": "search", "complete": "complete"}.get(str(수집방식).strip())
    if mode is None:
        raise ValueError("수집방식은 '검색' 또는 '전량' 중 하나로 적어 주세요.")
    queries = [q for q in (검색어 or []) if str(q).strip()]
    use_query = bool(queries)
    SEARCH_DELAY, SEARCH_WINDOW_DAYS = float(요청간격_초), 1

    print(f"수집 기간 {d0}~{d1} · 언론사 {len(offices)}곳 · "
          f"{'검색어 ' + ', '.join(queries) if use_query else '검색어 없음(언론사 전체 기사)'} · "
          f"방식 {'네이버 검색' if mode == 'search' and use_query else '전체 기사 확인'} · "
          f"댓글 {'댓글+답글 전수' if 댓글수집 else '수집 안 함'}")
    verify_offices(offices)
    collect_multi_keywords(
        queries=queries if use_query else ["전체기사"], news_offices=offices, base_path=저장폴더,
        start_date=d0, end_date=d1, max_pages=10000, collect_comments=댓글수집,
        collect_image=이미지수집, image_quality="low", separate_image_rows=False,
        strict_filter=엄격필터, keyword_workers=1, comment_workers=동시댓글작업,
        use_query=use_query, collect_mode=mode, complete_workers=32, detail_workers=8)


In [8]:
# ③ 실행
run_collection(검색어, 언론사, 시작일, 종료일, 저장폴더, 수집방식=수집방식, 댓글수집=댓글수집,
               요청간격_초=요청간격_초, 엄격필터=엄격필터, 이미지수집=이미지수집)

수집 기간 20190101~20190131 · 언론사 31곳 · 검색어 5.18, 5·18 · 방식 네이버 검색 · 댓글 댓글+답글 전수
✓ 언론사 31곳 코드 확인 완료
수집 키워드: ['5.18', '5·18']
처리 방식: 순차
수집 방식: 네이버 검색

[5.18] ======================================================================
[5.18]   키워드: [5.18]  폴더: C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18
[5.18] ======================================================================

[5.18] ▶ 경향신문: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\경향신문\articles.csv)

[5.18] ▶ 국민일보: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\국민일보\articles.csv)

[5.18] ▶ 동아일보: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\동아일보\articles.csv)

[5.18] ▶ 서울신문: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\서울신문\articles.csv)

[5.18] ▶ 조선일보: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\조선일보\articles.csv)

[5.18] ▶ 중앙일보 뉴스 수집 시작...
[5.18]   (이어하기: 31/31개 구간은 이미 받음)


[5.18] 링크 수집: 100%|##########| 31/31 [00:00<?, ?it/s]

[5.18] 총 0개의 뉴스 링크를 수집했습니다. (기간 20190101~20190131)
[5.18] ⚠️ 중앙일보에서 수집된 링크가 없습니다.

[5.18] ▶ 한겨레: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\한겨레\articles.csv)

[5.18] ▶ 한국일보: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\한국일보\articles.csv)

[5.18] ▶ 매일경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\매일경제\articles.csv)

[5.18] ▶ 서울경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\서울경제\articles.csv)

[5.18] ▶ 아시아경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\아시아경제\articles.csv)

[5.18] ▶ 조선비즈: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\조선비즈\articles.csv)

[5.18] ▶ 한국경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\한국경제\articles.csv)

[5.18] ▶ 헤럴드경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\헤럴드경제\articles.csv)

[5.18] ▶ 파이낸셜뉴스: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\파이낸셜뉴스\articles.csv)

[5.18] ▶ KB

[5.18] 링크 수집: 100%|##########| 31/31 [00:00<?, ?it/s]

[5.18] 총 0개의 뉴스 링크를 수집했습니다. (기간 20190101~20190131)
[5.18] ⚠️ 문화일보에서 수집된 링크가 없습니다.

[5.18] ▶ 세계일보: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5.18\세계일보\articles.csv)

[5.18] ★ 모든 뉴스 수집이 완료되었습니다! ★

[5.18] ▶ 댓글 전수 수집을 시작합니다...
[5.18]   [JTBC] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [KBS] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [MBC] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [MBN] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [SBS] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [TV조선] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [YTN] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [경향신문] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [국민일보] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [뉴스1] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [뉴시스] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [동아일보] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [매일경제] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [머니투데이] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [서울경제] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [서울신문] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [세계일보] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [아시아경제] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [연합뉴스] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [연합뉴스TV] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [이데일리] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [조선비즈] 이미 댓글 수집 완료 — 건너뜀
[5.18]   [조선일

[5·18] 링크 수집: 100%|##########| 31/31 [00:00<?, ?it/s]

[5·18] 총 0개의 뉴스 링크를 수집했습니다. (기간 20190101~20190131)
[5·18] ⚠️ 중앙일보에서 수집된 링크가 없습니다.

[5·18] ▶ 한겨레: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\한겨레\articles.csv)

[5·18] ▶ 한국일보: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\한국일보\articles.csv)

[5·18] ▶ 매일경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\매일경제\articles.csv)

[5·18] ▶ 서울경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\서울경제\articles.csv)

[5·18] ▶ 아시아경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\아시아경제\articles.csv)

[5·18] ▶ 조선비즈: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\조선비즈\articles.csv)

[5·18] ▶ 한국경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\한국경제\articles.csv)

[5·18] ▶ 헤럴드경제: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\헤럴드경제\articles.csv)

[5·18] ▶ 파이낸셜뉴스: 이미 수집됨 — 건너뜀 (C:\Users\com\Desktop\네이버뉴스_수집_test_2609301722\5·18\파이낸셜뉴스\articles.csv)

[5·18] ▶ KB